<p class="lead">This notebook closes the course, now in R. The lecture slides are a map; this text is the territory. Units 1–8 introduced methods; this unit asks what is done with them in health, how it is done honestly, and which obligations come with doing it. Its four subunits — data mining, medical image recognition, medical diagnostic systems, professional ethics — are four views of one object, the deployed pattern-recognition system. Its centre is a <strong>capstone project</strong> (Section 4.4): a complete diagnostic model on the Wisconsin breast-cancer data, built with the methodology of Supplement S1, calibrated, evaluated with a decision curve, audited for fairness and explained.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Describe KDD and CRISP-DM, name the four task families of data mining and place every method of Units 2–7 in one.</li>
<li>Define support, confidence and lift; run Apriori by hand; explain why mining many rules produces spurious ones and how a permutation null and a multiple-comparison correction expose them.</li>
<li>Use an isolation forest and a bag of words, and state the limits of both.</li>
<li>Classify an imaging task; list what makes medical images different; compute Dice and IoU; read the evidence base critically.</li>
<li>Trace computer-aided diagnosis from MYCIN and INTERNIST-1 to decision support; apply likelihood ratios and Fagan's nomogram.</li>
<li>Build, calibrate and evaluate a diagnostic model honestly — nested cross-validation, calibration, decision curve, cost-based threshold, subgroup check, model card — and name TRIPOD+AI, DECIDE-AI and CONSORT-AI.</li>
<li>Define the three fairness criteria and the impossibility result; audit and explain a model; describe synthetic data, federated learning and differential privacy; name the Chilean and international rules; apply a deployment checklist.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read it linearly once, executing each chunk, then return to Section 4.4 and change one thing at a time. Blue boxes carry key ideas, orange boxes pitfalls, purple boxes links to other units or to BGM324, green boxes exercises. Only base R, <code>stats</code>, <code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>randomForest</code>, <code>cluster</code>, <code>nnet</code>, <code>pROC</code> and <code>ggplot2</code> are used; all data ship with base R or the <code>mlbench</code> package. The vocabulary of Supplement S1 is assumed. In a fresh Google Colab R runtime, run once <code>install.packages(c("e1071","randomForest","rpart.plot","cluster","mlbench","pROC"))</code>. Where the Python edition of this unit used scikit-learn's <code>IsolationForest</code>, <code>mlxtend</code>/Apriori, a bag-of-words vectoriser and a federated-learning loop, the R edition implements each from scratch in base R, as the course prefers when a method is more instructive than a package call.</p></div>



In [ ]:
# Libraries used throughout. Seeds are fixed so every number quoted in the text is reproducible.
suppressPackageStartupMessages({
  library(MASS); library(class); library(e1071); library(rpart); library(rpart.plot)
  library(randomForest); library(cluster); library(nnet); library(pROC); library(mlbench); library(ggplot2)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"; PURPLE <- "#6a4c93"
cat(R.version.string, "| e1071", as.character(packageVersion("e1071")),
    "| randomForest", as.character(packageVersion("randomForest")),
    "| pROC", as.character(packageVersion("pROC")), "\n")



The printout records the R version and the versions of the three modelling packages the capstone leans on, so that a reader can reproduce every figure below.<sup class="c"><a href="#r36">[36]</a></sup>

## 1. Where the course has arrived

Every unit added a stage or a method to the pattern-recognition pipeline of Unit 1. The figure redraws that pipeline with the units attached to the stage they serve and the cross-cutting concerns — evaluation, ethics, deployment — as the frame around it.



In [ ]:
stages <- list(
  c("Data and\nrepresentation","U1 feature vectors,\ndata types; S0 toolkit", BLUE),
  c("Unsupervised\nstructure","U5 k-means, hierarchical,\nPCA, ICA", BLUE),
  c("Supervised\nlearning","U2 regression, first classifier\nU3 Bayes, SVM, k-NN\nU4 trees, random forest\nU6 neural networks\nU8 image classification", ORANGE),
  c("Optimisation\nand search","U7 genetic algorithms\n(and gradient descent, U2/U6)", GREEN),
  c("Decision and\ndeployment","U9 diagnostic systems,\nthresholds, model cards", PURPLE))
plot.new(); plot.window(xlim=c(0,10), ylim=c(0,6)); 
for (i in seq_along(stages)) {
  s <- stages[[i]]; x <- 0.3 + (i-1)*1.95; col <- s[3]
  rect(x, 2.0, x+1.7, 4.6, border=col, col=adjustcolor(col, 0.12), lwd=1.5)
  text(x+0.85, 4.35, s[1], col=col, font=2, cex=0.8)
  text(x+0.85, 3.0, s[2], cex=0.62)
  if (i < length(stages)) arrows(x+1.72, 3.3, x+1.93, 3.3, length=0.08, lwd=1.2)
}
rect(0.15, 0.7, 9.85, 1.6, col="#f1f1f1", border=GREY)
text(5, 1.15, "Evaluation (Supplement S1): split, cross-validation, confusion matrix, ROC-AUC, leakage - used at every stage", cex=0.72)
rect(0.15, 5.0, 9.85, 5.9, col="#f1f1f1", border=GREY)
text(5, 5.45, "Data mining process (2)  -  Ethics, fairness, privacy, governance (U1 and 5)  -  Reporting standards (4.6)", cex=0.72)
title("The course as one pipeline: the stages and the units that supply them", cex.main=0.95)



<p class="caption"><b>Figure 1.</b> The course as one pipeline: the five stages with the unit that studied each, and the concerns that apply at every stage. Section 4.4 walks through the whole figure once on one dataset.</p>

Three remarks organise the unit. The pipeline is run many times, not once — hence the process models of Section 2. The methods of Units 2–7 are tools for different <em>tasks</em> (Section 2.2). And Units 1–8 evaluated models against a label, whereas this unit evaluates them against a <em>decision</em> — a biopsy ordered or not — which is why calibration, decision curves and costs dominate Section 4.

<div class="box bridge"><h4>Laboratory 6 and Workshop 6</h4><p>For Laboratory 6 the capstone of Section 4.4 is the template and the checklist of Section 5.6 the marking scheme. For Workshop 6, Section 6 lists fifteen real systems with their evidence level, and Section 3.3 explains how to judge it.</p></div>

## 2. Data mining (9.1)

### 2.1 Definition and the KDD process

Fayyad, Piatetsky-Shapiro and Smyth defined <em>knowledge discovery in databases</em> (KDD) as "the non-trivial process of identifying valid, novel, potentially useful, and ultimately understandable patterns in data"; <strong>data mining</strong> is the step in which an algorithm extracts them.<sup class="c"><a href="#r1">[1]</a></sup> Every adjective is a requirement — valid on new data (S1), not already known, decision-relevant, checkable — and the definition anticipates Section 5 by two decades.

KDD runs selection → pre-processing → transformation (Units 1, 5) → data mining (Units 2–7) → interpretation and evaluation, with loops back. Industry formalised the cycle in 2000 as <strong>CRISP-DM</strong>: business understanding, data understanding, data preparation, modelling, evaluation, deployment. A 2021 review found it still the most used process model, usually with deployment omitted or sketched — with the consequences of Section 4.7.<sup class="c"><a href="#r2">[2]</a></sup> Both models say the same thing: most of the work is before and after the algorithm.



In [ ]:
kdd <- c("Selection","Pre-processing","Transformation","Data mining","Interpretation /\nevaluation")
crisp <- c("Business\nunderstanding","Data\nunderstanding","Data\npreparation","Modelling","Evaluation","Deployment")
par(mfrow=c(2,1), mar=c(1,1,2,1))
for (cfg in list(list(kdd,"KDD process (1996)",BLUE), list(crisp,"CRISP-DM (2000)",ORANGE))) {
  steps <- cfg[[1]]; nm <- cfg[[2]]; col <- cfg[[3]]
  plot.new(); plot.window(xlim=c(0,10), ylim=c(-0.6,2.0)); w <- 9.2/length(steps)
  for (i in seq_along(steps)) { x <- 0.4 + (i-1)*w
    rect(x, 0.5, x+w*0.82, 1.5, col=adjustcolor(col,0.15), border=col)
    text(x+w*0.41, 1.0, steps[i], cex=0.72)
    if (i < length(steps)) arrows(x+w*0.82, 1.0, x+w, 1.0, length=0.06, lwd=1) }
  arrows(0.4+(length(steps)-2)*w+w*0.4, 0.35, 0.6, 0.35, length=0.06, lwd=0.9, col=GREY)
  text(0.3, 1.8, nm, col=col, font=2, cex=0.85, adj=0)
}



<p class="caption"><b>Figure 2.</b> KDD and CRISP-DM. The algorithmic step is one box in five or six; the grey arrows are the feedback loops. Only CRISP-DM names deployment.</p>

### 2.2 The four task families, and where Units 2–7 fit

Data-mining tasks are grouped by <em>question</em>, not by algorithm, and the first decision in any project is which question is asked.

| Family | Question | Output | Methods in this course |
|---|---|---|---|
| **Classification** | Which category does this object belong to? | Class label or probability | U1 nearest centroid; U2 logistic regression; U3 Bayes, k-NN, SVM; U4 trees, forests; U6 MLP; U8 image classifiers |
| **Regression** | What value of a continuous quantity is expected? | A number | U2 linear and polynomial; k-NN, SVM, tree and MLP regression (U3, U4, U6) |
| **Clustering** | Which objects resemble one another? | Groups found without labels | U5 k-means, hierarchical, PCA, ICA |
| **Association** | Which events occur together? | Rules "if A then B" with a strength | Association rules and Apriori — **this unit only** |

Across the families cut <em>anomaly detection</em> (Section 2.7), <em>optimisation</em> (U7) and <em>text mining</em> (Section 2.8). No other unit covers association rules, so this section treats them in depth.

<div class="box bridge"><h4>Bridge to bioinformatics: the lecturer's research as data mining</h4><p>Two papers from the course's laboratory are KDD applied to chemistry. Morales and colleagues represented urease inhibitors and inactive compounds by physicochemical descriptors, trained classifiers including the tree ensembles of Unit 4, and derived <em>filter criteria</em> a chemist can apply before synthesis.<sup class="c"><a href="#r8">[8]</a></sup> González and colleagues did the same for TRPV1 inhibitors, added a repurposing screen of approved drugs, and validated candidates by docking and molecular-dynamics simulation.<sup class="c"><a href="#r9">[9]</a></sup> Objects are molecules, features are descriptors, the label is activity, and "deployment" is the choice of compounds to test: the QSAR models of BGM324 as pattern recognition.</p></div>

### 2.3 Association rules: support, confidence, lift

Agrawal, Imieliński and Swami introduced association rules in 1993 for the <em>market-basket</em> problem.<sup class="c"><a href="#r3">[3]</a></sup> In health the basket is a patient, the items are diagnoses, medications or laboratory abnormalities, and the rules are comorbidity and prescribing patterns or candidate adverse reactions. Let \(D\) be a database of \(N\) <strong>transactions</strong>, each a subset of the items \(I\); an <strong>itemset</strong> is any subset of \(I\) and a <strong>rule</strong> \(A \Rightarrow B\) an implication between disjoint itemsets. Three quantities describe a rule:

<div class="eq">\[\text{supp}(A) = \frac{|\{T \in D : A \subseteq T\}|}{N}, \qquad \text{conf}(A \Rightarrow B) = \frac{\text{supp}(A \cup B)}{\text{supp}(A)}, \qquad \text{lift}(A \Rightarrow B) = \frac{\text{conf}(A \Rightarrow B)}{\text{supp}(B)} = \frac{\text{supp}(A \cup B)}{\text{supp}(A)\,\text{supp}(B)}\]
<span class="where">Support estimates \(P(A)\) (for a rule, \(\text{supp}(A \cup B)\)); confidence estimates \(P(B \mid A)\); lift is \(P(A, B)/(P(A)P(B))\) — 1 under independence.</span></div>

Confidence alone misleads: if 90 % of patients take paracetamol, "hypertension ⇒ paracetamol" has confidence 0.9 and says nothing. Brin, Motwani and Silverstein showed in 1997 that confidence can rank rules in the reverse order of their dependence and proposed measuring departure from independence instead — lift (<em>interest</em>) and, as a formal test, the chi-squared statistic of Section 2.6.<sup class="c"><a href="#r5">[5]</a></sup>

### 2.4 The Apriori algorithm, by hand

There are \(2^{|I|}\) itemsets, so enumeration is impossible. Agrawal and Srikant's <strong>Apriori</strong> (1994) exploits the <em>downward-closure</em> property: <strong>every subset of a frequent itemset is frequent</strong>, so an itemset can be frequent only if all its subsets are.<sup class="c"><a href="#r4">[4]</a></sup> Level by level: (1) keep single items above minimum support, \(L_1\); (2) <strong>join</strong> pairs of frequent \((k-1)\)-itemsets sharing their first \(k-2\) items into candidates \(C_k\); (3) <strong>prune</strong> candidates with a \((k-1)\)-subset not in \(L_{k-1}\); (4) count the survivors in one pass, giving \(L_k\); repeat until empty; (5) output every split \(A \Rightarrow B\) of a frequent itemset whose confidence passes a minimum, with its lift.

**Worked example.** Six patients, five items in the order HTN < DM < Met < ACEi < Statin (hypertension, type 2 diabetes, metformin, ACE inhibitor, statin); minimum support 2 of 6.

| patient | items |
|---|---|
| 1 | HTN, DM, Met, Statin |
| 2 | HTN, ACEi |
| 3 | DM, Met |
| 4 | HTN, DM, Met, ACEi |
| 5 | HTN, Statin |
| 6 | DM, Met, Statin |

**Level 1.** HTN 4, DM 4, Met 4, ACEi 2, Statin 3 — all five frequent. **Level 2.** The join gives all \(\binom{5}{2} = 10\) pairs (nothing to prune, every 1-subset is frequent); counting leaves {DM, ACEi}, {Met, ACEi} and {ACEi, Statin} below support. **Level 3, join:** pairs of \(L_2\) with the same first item give seven candidates. **Prune:** {HTN, DM, ACEi} contains the infrequent {DM, ACEi}; {HTN, Met, ACEi} contains {Met, ACEi}; {HTN, ACEi, Statin} contains {ACEi, Statin} — three pruned without counting. **Count** the four survivors: {HTN, DM, Met} in patients 1 and 4 (frequent); {HTN, DM, Statin} and {HTN, Met, Statin} in patient 1 only (not); {DM, Met, Statin} in 1 and 6 (frequent). \(L_3\) has two itemsets sharing no 2-prefix, so \(C_4\) is empty and the algorithm stops. **Rules:** DM ⇒ Met has support 4/6 = 0.67, confidence 4/4 = 1.00 and lift 1.00/(4/6) = 1.50; HTN ⇒ Statin has confidence 2/4 = 0.50 and lift 0.50/(3/6) = 1.00 — no association at all. The chunk below reproduces the trace from a base-R Apriori written for teaching.



In [ ]:
# A minimal Apriori on a logical transaction matrix (base R, no package): join + prune + count, level by level.
apriori <- function(M, min_support, verbose=FALSE) {
  items <- colnames(M); ord <- function(v) v[order(match(v, items))]
  key <- function(v) paste(ord(v), collapse="|")
  support <- list(); counted <- integer(); L <- list()
  for (j in seq_along(items)) if (mean(M[,j]) >= min_support) L[[length(L)+1]] <- items[j]
  for (c in L) support[[key(c)]] <- mean(M[, c]); k <- 1
  Lkeys <- function(LL) vapply(LL, key, "")
  while (length(L) > 0) {
    k <- k + 1; Lset <- Lkeys(L); C <- list(); seen <- character()
    for (a in L) for (b in L) {                                          # JOIN: same (k-2)-prefix
      if (identical(a[-length(a)], b[-length(b)]) && match(a[length(a)], items) < match(b[length(b)], items)) {
        cand <- ord(c(a, b[length(b)]))                                  # PRUNE: all (k-1)-subsets frequent
        ok <- all(vapply(seq_along(cand), function(d) key(cand[-d]) %in% Lset, TRUE)); kk <- key(cand)
        if (ok && !(kk %in% seen)) { seen <- c(seen, kk); C[[length(C)+1]] <- cand } } }
    counted[as.character(k)] <- length(C); newL <- list()
    if (length(C)) C <- C[order(vapply(C, function(cc) paste(sprintf("%02d", match(cc, items)), collapse=""), ""))]
    for (cand in C) {                                                    # COUNT survivors in one pass
      s <- mean(rowSums(M[, cand, drop=FALSE]) == length(cand))
      if (verbose) cat(sprintf("  L%d {%s}: supp %.2f -> %s\n", k, paste(cand,collapse=", "), s, ifelse(s>=min_support,"frequent","discarded")))
      if (s >= min_support) { support[[key(cand)]] <- s; newL[[length(newL)+1]] <- cand } }
    L <- newL }
  list(support=support, counted=counted, items=items) }

toy <- rbind(c(1,1,1,0,1),c(1,0,0,1,0),c(0,1,1,0,0),c(1,1,1,1,0),c(1,0,0,0,1),c(0,1,1,0,1)) == 1
colnames(toy) <- c("HTN","DM","Met","ACEi","Statin")
res <- apriori(toy, 2/6, verbose=TRUE)
cat("candidates counted per level:", paste(names(res$counted), res$counted, sep="="), "\n")
cat("frequent 3-itemsets:", paste(gsub("\\|", ",", names(res$support)[lengths(strsplit(names(res$support),"\\|"))==3]), collapse=" ; "), "\n")



The trace matches the hand computation exactly: level 2 counts all 10 pairs (nothing to prune), level 3 counts only the 4 survivors of pruning, and the two frequent 3-itemsets are {HTN, DM, Met} and {DM, Met, Statin}; \(C_4\) is empty. The next chunk turns frequent itemsets into rules.



In [ ]:
# Every split A => B of a frequent itemset, with support, confidence and lift.
rules_from <- function(result, min_conf=0.0) {
  support <- result$support; items <- result$items
  ordkey <- function(v) paste(v[order(match(v, items))], collapse="|"); rows <- list()
  for (S in names(support)) {
    SI <- strsplit(S, "\\|")[[1]]; if (length(SI) < 2) next; s <- support[[S]]
    for (r in 1:(length(SI)-1)) { cb <- combn(SI, r)
      for (ci in seq_len(ncol(cb))) { A <- cb[,ci]; B <- setdiff(SI, A)
        conf <- s/support[[ordkey(A)]]
        rows[[length(rows)+1]] <- data.frame(antecedent=paste(A[order(match(A,items))],collapse=" & "),
          consequent=paste(B[order(match(B,items))],collapse=" & "), support=s, confidence=conf,
          lift=conf/support[[ordkey(B)]], stringsAsFactors=FALSE) } } }
  out <- do.call(rbind, rows); out <- out[out$confidence>=min_conf,]; out[order(-out$lift),] }

toy_rules <- rules_from(res)
sel <- toy_rules[toy_rules$antecedent %in% c("DM","HTN") & toy_rules$consequent %in% c("Met","Statin"),]
sel[,c("support","confidence","lift")] <- round(sel[,c("support","confidence","lift")], 2)
print(sel, row.names=FALSE)



The computed rules confirm the hand values: DM ⇒ Met has support 0.67, confidence 1.00 and lift 1.50, whereas HTN ⇒ Statin has confidence 0.50 and lift 1.00 — the same support as several others and no association at all. The same code runs unchanged on the 1,500-patient table below.

<div class="exercise"><h4>Exercise 1</h4><p>Lower the minimum support to 1/6 and re-run both chunks: how many level-3 candidates are counted now, and how many rules result? Then add a seventh patient with items {ACEi, Statin} at support 2/6: which previously pruned level-3 candidate is now counted, and does it become frequent?</p></div>

### 2.5 Apriori on a synthetic clinical database

Real records cannot ship with a notebook, so the next chunk generates 1,500 patients with seven comorbidities and eight medications, planting plausible associations — metformin for diabetics, ACE inhibitors for hypertensives, statins for both, salbutamol for COPD, SSRIs for depression, beta-blockers for heart failure, obesity raising diabetes and hypertension raising heart failure — and leaving everything else independent, so that we can see which rules the miner recovers and which it invents.



In [ ]:
# A synthetic 'electronic record' of 1,500 patients: comorbidities and medications with planted associations.
set.seed(0); n_pat <- 1500
bern <- function(p, size=n_pat) runif(size) < p
htn <- bern(0.45); obes <- bern(0.30); copd <- bern(0.12); dep <- bern(0.15); ckd <- bern(0.08)
dm <- bern(0.25) | (obes & bern(0.25))                      # obesity raises the probability of diabetes
hf <- bern(0.10) | (htn & bern(0.10))                       # hypertension raises the probability of heart failure
prescribed <- function(cond, p_if, p_else) ifelse(cond, bern(p_if), bern(p_else))
records <- data.frame(
  hypertension=htn, type2_diabetes=dm, obesity=obes, COPD=copd, heart_failure=hf, depression=dep, CKD=ckd,
  metformin=prescribed(dm,0.75,0.02), insulin=prescribed(dm,0.25,0.01), ACE_inhibitor=prescribed(htn,0.55,0.05),
  beta_blocker=prescribed(hf,0.70,0.08), statin=prescribed(dm & htn,0.70,0.0) | prescribed(dm | htn,0.35,0.08),
  salbutamol=prescribed(copd,0.65,0.04), SSRI=prescribed(dep,0.60,0.03), aspirin=prescribed(hf | htn,0.30,0.10))
M <- as.matrix(records)
cat(sprintf("%d patients x %d items; items per patient: median %.0f, max %d\n",
    nrow(M), ncol(M), median(rowSums(M)), max(rowSums(M))))
print(round(sort(colMeans(M), decreasing=TRUE), 3))



The item supports range from hypertension at about 0.47 down to chronic kidney disease at about 0.07, and a typical patient carries three items. The planted dependences are invisible in these marginal frequencies — they live in the joint distribution, which is what Apriori searches.



In [ ]:
# Apriori at minimum support 0.05, then the rules with confidence >= 0.5, sorted by lift.
t0 <- Sys.time(); res_rec <- apriori(M, min_support=0.05)
by_level <- sapply(1:5, function(k) sum(lengths(strsplit(names(res_rec$support),"\\|"))==k))
cat(sprintf("%.2f s | frequent itemsets per level (1..5): %s | candidates counted: %s\n",
    as.numeric(difftime(Sys.time(),t0,units="secs")), paste(by_level,collapse=" "),
    paste(names(res_rec$counted), res_rec$counted, sep="=", collapse=" ")))
rec_rules <- rules_from(res_rec, min_conf=0.5)
cat(nrow(rec_rules), "rules with confidence >= 0.5; the 8 with the highest lift:\n")
top <- head(rec_rules, 8); top[,c("support","confidence","lift")] <- round(top[,c("support","confidence","lift")], 3)
print(top, row.names=FALSE)



The miner recovers the planted structure in a few hundredths of a second. The highest lifts, about 5 to 6, belong to the drug–indication pairs (depression ⇔ SSRI, heart failure ⇔ beta-blocker) and to compound rules over hypertension, diabetes, metformin, ACE inhibitor and statin, whose two-item consequents are rare. Each association appears in both directions with equal support and lift but different confidence — "SSRI ⇒ depression" is more confident than "depression ⇒ SSRI" because not every depressed patient is treated but almost every SSRI user is depressed; lift is symmetric, confidence is not, neither is causal. Rules over compound itemsets are largely redundant with the pairwise rules that compose them.

### 2.6 The pitfall: spurious rules and multiple comparisons

A miner tests every pair and every larger itemset: 105 pairs for 15 items, over a million for 1,500 ICD-10 codes. Some will have lift well above 1 by chance, and the miner, which has no null hypothesis, reports them beside the real ones. The chunk quantifies this by <em>destroying every association</em> — permuting each column independently, which keeps item frequencies but makes every item independent of every other — and mining the permuted table at a lower minimum support, where rare pairs are noisy.



In [ ]:
# Permutation null: shuffle each column independently (associations destroyed, frequencies kept) and mine again.
pairwise <- function(rt) { r <- rt[!grepl("&", rt$antecedent) & !grepl("&", rt$consequent),]; r[r$antecedent < r$consequent,] }
real_pairs <- pairwise(rules_from(apriori(M, 0.02)))
set.seed(0); null_max <- numeric(); null_lifts <- numeric()
for (rep in 1:50) { sh <- apply(M, 2, sample)
  pr <- pairwise(rules_from(apriori(sh, 0.02))); null_lifts <- c(null_lifts, pr$lift); null_max <- c(null_max, max(pr$lift)) }
cat(sprintf("real table: %d pairwise rules, %d with lift > 1.3, maximum lift %.2f\n",
    nrow(real_pairs), sum(real_pairs$lift > 1.3), max(real_pairs$lift)))
cat(sprintf("50 shuffled tables: maximum pairwise lift per table ranges %.2f-%.2f; %.1f%% of null rules have lift > 1.3\n",
    min(null_max), max(null_max), 100*mean(null_lifts > 1.3)))
bins <- seq(0.3, 2.2, length.out=39)
hist(null_lifts, breaks=bins, freq=FALSE, col=adjustcolor(GREY,0.7), border="white",
     main="Lift of rules with no association behind them (grey) spreads well beyond 1", xlab="lift", cex.main=0.9)
hist(pmin(real_pairs$lift, 2.2), breaks=bins, freq=FALSE, col=adjustcolor(ORANGE,0.6), border="white", add=TRUE)
abline(v=1, lty=2); legend("topright", c("null (shuffled tables)","real table (clipped at 2.2)"),
       fill=c(adjustcolor(GREY,0.7), adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figure 3.</b> Lift of pairwise rules from 50 shuffled copies of the table (grey; every association destroyed) against the real table (orange; planted lifts up to about 5.5 clipped at 2.2). The null distribution is centred on 1 but reaches about 1.5–1.7: whoever reports every rule with lift above 1.3 reports rules from the grey distribution.</p>

The remedy is that of Unit 3 and Supplement S1: treat each rule as a hypothesis test — for a pair, the chi-squared test of independence on its 2 × 2 table — and correct for the \(m\) tests performed, with <strong>Bonferroni</strong> (\(p < \alpha/m\)) or the Benjamini–Hochberg false-discovery rate. The chunk below does so for the pairwise rules of the real table.



In [ ]:
# Chi-squared test of independence for every pairwise rule, with and without a Bonferroni correction.
chi2_p <- function(a, b) suppressWarnings(chisq.test(table(records[[a]], records[[b]]))$p.value)
tested <- real_pairs; tested$p_value <- mapply(chi2_p, tested$antecedent, tested$consequent); m <- nrow(tested)
cat(sprintf("%d pairwise rules tested | p < 0.05: %d | Bonferroni p < 0.05/%d: %d\n",
    m, sum(tested$p_value < 0.05), m, sum(tested$p_value < 0.05/m)))
ov <- tested[tested$lift > 1.1 & tested$lift < 1.6,]; ov <- ov[order(-ov$lift),]
ov$sig_05 <- ov$p_value < 0.05; ov$sig_bonf <- ov$p_value < 0.05/m
ov$support <- round(ov$support,3); ov$lift <- round(ov$lift,3); ov$p_value <- signif(ov$p_value,2)
print(head(ov[,c("antecedent","consequent","support","lift","p_value","sig_05","sig_bonf")], 8), row.names=FALSE)



Of 81 pairwise rules, 27 pass \(p < 0.05\) and 21 survive the stricter Bonferroni threshold; the six that pass one and fail the other are exactly the weak, borderline rules of the overlap region. The planted rules (obesity ⇒ diabetes, hypertension ⇒ statin, aspirin ⇒ hypertension) survive with \(p\)-values that are numerically zero. Three limits remain: a surviving association may be trivial; it may be <em>confounded</em>, as metformin ⇒ obesity is (it runs through diabetes); and no rule is causal — mining generates hypotheses that epidemiological designs test. Pharmacovigilance uses exactly this pipeline: disproportionality measures, statistical screening, clinical review.

<div class="box warn"><h4>The look-elsewhere effect</h4><p>Every method that <em>searches</em> — rules, anomalies, subgroups (Section 5.2), feature importances (Section 5.3) — tests many hypotheses at once, and the maximum of many noisy estimates is biased upwards, as S1 showed for hyper-parameter selection. Any quantity selected because it was the largest must be re-estimated on data that played no part in the selection.</p></div>

### 2.7 Anomaly detection in one chunk

Anomaly detection asks <em>which objects are unlike the rest?</em> and needs no labels — the question for equipment faults, fraudulent claims, laboratory values of an unrecognised disease or data-entry errors. The <strong>isolation forest</strong> of Liu, Ting and Zhou reuses the trees of Unit 4: random splits until every point is isolated, and the <em>number of splits needed</em> is the score — anomalies, few and different, are isolated quickly.<sup class="c"><a href="#r6">[6]</a></sup> R has no isolation forest in the course packages, so the chunk implements one in about twenty lines of base R, trains it on the <em>benign</em> Wisconsin cases only, as if malignancy were an unseen condition, and asks whether malignant cases are flagged.



In [ ]:
# The Wisconsin breast-cancer data from mlbench: the R stand-in for scikit-learn's load_breast_cancer.
data(BreastCancer, package="mlbench")
bc <- BreastCancer[,-1]                                  # drop the Id column
bc <- bc[complete.cases(bc),]                            # drop the 16 rows with missing values
bc[,1:9] <- lapply(bc[,1:9], function(z) as.numeric(as.character(z)))   # ordinal factor -> numeric
feat <- as.matrix(bc[,1:9]); yc <- as.integer(bc$Class == "malignant")  # yc: 1 = malignant (the 'disease' coding)
cat("rows x features:", nrow(feat), "x", ncol(feat), "| malignant:", sum(yc), "of", length(yc), "\n")



The curated table has 683 aspirates described by nine ordinal nuclear-morphology scores, 239 of them malignant.<sup class="c"><a href="#r37">[37]</a></sup> It carries nine features rather than the thirty of the scikit-learn copy, but it is the same diagnostic problem and the same source.



In [ ]:
# Isolation forest from scratch: random split trees; anomaly score = 2^(-E[path length]/c(n)).
c_factor <- function(n) if (n > 2) 2*(log(n-1)+0.5772156649) - 2*(n-1)/n else if (n==2) 1 else 0
itree <- function(X, depth, md) { n <- nrow(X)
  if (depth >= md || n <= 1) return(list(leaf=TRUE, size=n))
  f <- sample(ncol(X), 1); rg <- range(X[,f]); if (rg[1]==rg[2]) return(list(leaf=TRUE, size=n))
  sp <- runif(1, rg[1], rg[2]); lft <- X[,f] < sp
  list(leaf=FALSE, f=f, sp=sp, L=itree(X[lft,,drop=FALSE],depth+1,md), R=itree(X[!lft,,drop=FALSE],depth+1,md)) }
path_len <- function(x, nd, d=0) if (nd$leaf) d + c_factor(nd$size) else if (x[nd$f] < nd$sp) path_len(x,nd$L,d+1) else path_len(x,nd$R,d+1)
iforest <- function(X, n_trees=200, ss=256) { X <- as.matrix(X); n <- nrow(X); ss <- min(ss,n); md <- ceiling(log2(ss))
  list(trees=lapply(1:n_trees, function(t) itree(X[sample(n,ss),,drop=FALSE],0,md)), cn=c_factor(ss)) }
score_if <- function(mdl, X) { X <- as.matrix(X); 2^(-rowMeans(sapply(mdl$trees, function(tr) apply(X,1,path_len,nd=tr)))/mdl$cn) }

set.seed(0); benign_only <- feat[yc==0,]
iso <- iforest(benign_only); score <- score_if(iso, feat)
auc_anom <- as.numeric(auc(yc, score, direction="<", quiet=TRUE))
cutoff <- quantile(score_if(iso, benign_only), 0.90)     # operating point: flag what is more anomalous than 90% of normals
flag <- score >= cutoff
cat(sprintf("trained on %d benign cases | AUC of the anomaly score for malignancy: %.3f\n", nrow(benign_only), auc_anom))
cat(sprintf("flagged as anomalous: %.1f%% of malignant, %.1f%% of benign\n", 100*mean(flag[yc==1]), 100*mean(flag[yc==0])))
hist(score[yc==0], breaks=30, col=adjustcolor(BLUE,0.7), border="white", xlim=range(score),
     main="Isolation forest: malignant aspirates are isolated faster", xlab="anomaly score", ylab="cases", cex.main=0.9)
hist(score[yc==1], breaks=30, col=adjustcolor(ORANGE,0.7), border="white", add=TRUE)
legend("topright", c("benign (seen in training)","malignant (never seen)"), fill=c(adjustcolor(BLUE,0.7),adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figure 4.</b> Anomaly scores of the 683 aspirates from an isolation forest that saw only benign cases. Malignant cases (orange) are isolated in fewer splits and the score reaches an AUC of about 0.99 without a single malignant label. The operating point — flag anything more anomalous than 90 % of the normal training data — catches almost every malignant case while raising a false alarm on about 10 % of benign ones by construction.</p>

On these nine clean ordinal features the detector separates the classes more sharply than the thirty-feature scikit-learn copy did (there the AUC was about 0.96 with substantial overlap), but the lesson is unchanged: a supervised model on the same data reaches a comparable AUC (the capstone of Section 4.4 reaches about 0.997), and the price of having no labels is that the detector also flags instrument faults and rare benign conditions. Its role is screening for the unexpected and quality control; its output is a ranked list for a human, not a diagnosis — <em>different</em> is not <em>diseased</em>.

### 2.8 Text mining in one paragraph

Most clinical information is free text. The simplest way into the data matrix of Unit 1 is the <strong>bag of words</strong>: each document becomes a vector of word counts over a shared vocabulary, order ignored; naive Bayes, SVM or clustering follow. The representation does not know that "no fever" negates "fever" or that "MI" is "myocardial infarction"; clinical natural-language processing is largely the repair of such defects — negation detection, abbreviation expansion, mapping to SNOMED CT, contextual embeddings from language models. A review of 263 clinical information-extraction studies found bag-of-words and rule-based methods still dominant, most systems single-institution, few externally validated.<sup class="c"><a href="#r7">[7]</a></sup> The chunk shows the idea on six synthetic sentences with a multinomial naive Bayes written from scratch.



In [ ]:
# Bag of words: six synthetic clinical notes become a 6 x |vocabulary| count matrix; a multinomial naive Bayes learns from it.
notes <- c("patient reports chest pain on exertion and shortness of breath",
           "productive cough for three weeks with fever and night sweats",
           "chest pain radiating to left arm, diaphoresis, nausea",
           "persistent cough, wheeze, history of smoking, no fever",
           "exertional chest tightness relieved by rest", "cough with fever and pleuritic chest pain")
label <- c("cardiac","respiratory","cardiac","respiratory","cardiac","respiratory")
stop_en <- c("on","and","for","with","to","of","by","the","a","an","no","history")
tok <- function(s) { w <- unlist(strsplit(tolower(gsub("[^a-z ]"," ",s)), "\\s+")); w[nchar(w)>0 & !(w %in% stop_en)] }
toks <- lapply(notes, tok); vocab <- sort(unique(unlist(toks)))
bow <- t(sapply(toks, function(t) table(factor(t, levels=vocab)))); rownames(bow) <- paste0("note ", seq_along(notes))
mnb_fit <- function(X,y,alpha=1) { cl <- unique(y)
  list(cl=cl, lp=setNames(log(sapply(cl,function(c) mean(y==c))),cl),
       ll=sapply(cl, function(c){ cnt <- colSums(X[y==c,,drop=FALSE])+alpha; log(cnt/sum(cnt)) })) }
mnb_pred <- function(m,X) { sc <- sweep(X %*% m$ll, 2, m$lp, "+")
  list(pred=m$cl[apply(sc,1,which.max)], prob=t(apply(sc,1,function(r){e<-exp(r-max(r)); e/sum(e)}))) }
nb <- mnb_fit(bow, label); new_note <- "sudden chest pain with sweating"
pr <- mnb_pred(nb, matrix(table(factor(tok(new_note),levels=vocab)), nrow=1))
cat(sprintf("vocabulary of %d words (stop-words removed); matrix %d x %d\n", length(vocab), nrow(bow), ncol(bow)))
cat("new note:", new_note, "->", pr$pred, "| P(cardiac) =", round(pr$prob[1, which(nb$cl=="cardiac")], 2), "\n")
print(bow[, c("chest","pain","cough","fever","exertion","sweats")])



Six sentences became a count matrix over 27 words, and the multinomial naive Bayes of Unit 3 learned that "chest" and "pain" point to a cardiac note and "cough" and "fever" to a respiratory one, so that the new note is classified cardiac with probability about 0.73. The failure is already visible: note 6 contains "chest pain" (pleuritic, respiratory) and note 4 a negated "fever", yet both count as present. The classifier is a toy; the representation is the one the first generation of clinical NLP used at scale, with these limits.

<div class="exercise"><h4>Exercise 2</h4><p>Add two notes that use negation ("no chest pain", "denies fever") with appropriate labels and re-run: does the prediction for the new note change? Then form word-pair features by concatenating adjacent tokens (a bigram vectoriser) so that "no fever" becomes one feature, and inspect how many more columns the vocabulary gains. What does this fix cost in the number of features?</p></div>

## 3. Medical image recognition (9.2)

Unit 8 did the technical work of this subunit; this section is a synthesis: the tasks, why medical images are hard, how a segmentation is measured, what the evidence says.

### 3.1 The task taxonomy

| Task | Input → output | Clinical example | Typical metric |
|---|---|---|---|
| **Classification** | Whole image → label or probability | Referable diabetic retinopathy from a fundus photograph; malignant vs benign skin lesion | Sensitivity, specificity, ROC-AUC (S1) |
| **Detection** | Image → bounding boxes or points | Lung nodules on CT; microcalcifications on a mammogram | Sensitivity at a given number of false positives per image |
| **Segmentation** | Image → a label per pixel/voxel | Tumour volume on MRI; ventricle contour; nuclei in histology | Dice, IoU (Section 3.4) |
| **Registration** | Two images → aligning transform | Pre- and post-operative scans; atlas labelling; motion correction | Target registration error (mm) |
| **Reconstruction** | Raw or degraded data → image | Undersampled MRI; low-dose CT denoising | SNR, structural similarity |

The Wisconsin data sit upstream of all five: their nuclear-morphology features were extracted from nuclei first <em>segmented</em> in a cytology image.

### 3.2 What makes medical images different

Deep learning for images was developed on ImageNet: 1.2 million photographs with labels anyone can supply. Medical imaging differs in every respect, and each difference is a failure mode of a method transferred naively: **small datasets** (networks overfit; transfer learning and augmentation, Unit 8, are partial remedies); **scarce, noisy labels** (a radiologist's time, a biopsy, an outcome years later; inter-reader disagreement gives the ground truth its own error rate); **three and four dimensions** (large data per case, few cases); **class imbalance** (screening prevalence below 1 %, where accuracy is meaningless); **domain shift** (manufacturer, protocol, population — Section 3.5); **subtle, local signal** (a few dozen pixels in megapixels); **high stakes** (the clinician must be able to check the output — Section 5.3).

### 3.3 The evidence base

Liu and colleagues pooled 82 studies comparing deep learning with health-care professionals on the same images: pooled sensitivity about 87 % and specificity about 93 % for the models against 86 % and 91 % for clinicians — equivalence on average — but only 25 studies validated out of sample and only 14 compared both on the same external sample.<sup class="c"><a href="#r10">[10]</a></sup> Nagendran and colleagues examined design: of 81 non-randomised studies few were prospective, most compared the model with a handful of clinicians reading isolated images, data and code were rarely available, abstracts often over-claimed, and only two randomised trials existed.<sup class="c"><a href="#r11">[11]</a></sup> The problem is not that the models do not work but that the paper usually cannot tell us whether they do.

<div class="box key"><h4>How to read a performance claim</h4><p>Ask, in order: (1) external validation, or a hold-out from the same pool? (2) clinicians compared under realistic conditions, or reading isolated images? (3) at what prevalence, and what does the specificity mean in false positives at the prevalence of use? (4) ground truth a biopsy or outcome, or another reader's opinion? (5) a prospective study, and a randomised trial of the <em>decision</em>? Most published systems fail (1); almost all fail (5).</p></div>

### 3.4 Segmentation metrics: Dice and IoU

A predicted mask \(S\) is compared with a reference \(G\). Pixel accuracy is useless — a tumour occupying 2 % of the image gives 98 % accuracy to an empty mask — so overlap measures are used. The <strong>Dice coefficient</strong>, named after the ecologist L. R. Dice, who proposed it in 1945 for species co-occurrence, and the <strong>intersection over union</strong> (IoU, Jaccard index) are

<div class="eq">\[\text{Dice}(S, G) = \frac{2\,|S \cap G|}{|S| + |G|}, \qquad \text{IoU}(S, G) = \frac{|S \cap G|}{|S \cup G|}, \qquad \text{Dice} = \frac{2\,\text{IoU}}{1 + \text{IoU}}\]
<span class="where">Both range from 0 to 1 and rank segmentations identically; Dice is the F1-score of Supplement S1 applied to pixels.</span></div>



In [ ]:
# Dice and IoU on a synthetic mask pair: a circular 'lesion' (truth) and a shifted, enlarged prediction.
g <- expand.grid(x=0:63, y=0:63)
truth <- (g$x-30)^2 + (g$y-32)^2 <= 10^2                  # reference: radius 10 centred at (30, 32)
pred  <- (g$x-34)^2 + (g$y-30)^2 <= 11^2                  # prediction: shifted by (4, -2), radius 11
inter <- sum(truth & pred); union <- sum(truth | pred)
dice <- 2*inter/(sum(truth)+sum(pred)); iou <- inter/union
cat(sprintf("|G| = %d, |S| = %d, |S & G| = %d, |S | G| = %d\n", sum(truth), sum(pred), inter, union))
cat(sprintf("Dice = %.3f, IoU = %.3f, 2*IoU/(1+IoU) = %.3f  |  pixel accuracy = %.3f\n", dice, iou, 2*iou/(1+iou), mean(truth==pred)))
mk <- function(v) matrix(v, 64, 64)
ov <- array(1, c(64,64,3))
TT <- mk(truth); PP <- mk(pred)
ov[,,1][TT & PP] <- 0.3; ov[,,2][TT & PP] <- 0.6; ov[,,3][TT & PP] <- 0.3
ov[,,1][TT & !PP] <- 0.2; ov[,,2][TT & !PP] <- 0.4; ov[,,3][TT & !PP] <- 0.8
ov[,,1][!TT & PP] <- 0.9; ov[,,2][!TT & PP] <- 0.5; ov[,,3][!TT & PP] <- 0.2
par(mfrow=c(1,3), mar=c(1,1,2,1))
image(mk(truth), col=c("white",BLUE), axes=FALSE, main="reference mask G")
image(mk(pred),  col=c("white",ORANGE), axes=FALSE, main="predicted mask S")
plot(as.raster(aperm(ov, c(2,1,3)))); title(sprintf("overlap: Dice %.2f, IoU %.2f", dice, iou))



<p class="caption"><b>Figure 5.</b> A reference mask, a prediction displaced by four pixels and slightly too large, and their overlap (green both, blue missed, orange over-segmented). Pixel accuracy is about 0.95 because most of the image is background, yet Dice is about 0.73 and IoU about 0.57. Inter-expert Dice is about 0.9 for well-defined organs and 0.7 or lower for diffuse lesions, which bounds what any model can be shown to achieve.</p>

### 3.5 A complete small image pipeline, as a reminder

R has no digits dataset, so the chunk <em>simulates</em> one: six 8 × 8 patterns (a ring, a bar, a seven, a cross, an L and an X), each drawn 170 times with random amplitude and pixel noise, the border pixels left near zero on purpose. The whole pipeline of Figure 1 then runs on them — stratified split, standardisation, PCA (Unit 5), a support vector machine (Unit 3) and a multilayer perceptron (Unit 6) — as a rehearsal for Section 4.4.



In [ ]:
# Simulated 8x8 'digit' images (six classes, zero border); a reminder that R has no load_digits.
templates <- list(
  zero = c(0,0,0,0,0,0,0,0, 0,0,1,1,1,1,0,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,0,1,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  one  = c(0,0,0,0,0,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  seven= c(0,0,0,0,0,0,0,0, 0,1,1,1,1,1,0,0, 0,0,0,0,0,1,0,0, 0,0,0,0,1,0,0,0, 0,0,0,1,0,0,0,0, 0,0,1,0,0,0,0,0, 0,0,1,0,0,0,0,0, 0,0,0,0,0,0,0,0),
  plus = c(0,0,0,0,0,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,1,1,1,1,1,1,0, 0,1,1,1,1,1,1,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,0,0,0,0,0),
  ell  = c(0,0,0,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  ex   = c(0,0,0,0,0,0,0,0, 0,1,0,0,0,0,1,0, 0,0,1,0,0,1,0,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,0,1,0,0,1,0,0, 0,1,0,0,0,0,1,0, 0,0,0,0,0,0,0,0))
set.seed(0); per <- 170
X <- do.call(rbind, lapply(templates, function(b) t(sapply(1:per, function(i)
  pmin(pmax(b*12*runif(1,0.55,1.15) + rnorm(64,0,2.4), 0), 16)))))
yd <- factor(rep(names(templates), each=per), levels=names(templates))
idx <- unlist(lapply(levels(yd), function(c){ i<-which(yd==c); sample(i, round(0.7*length(i))) }))
Xtr <- X[idx,]; ytr <- yd[idx]; Xte <- X[-idx,]; yte <- yd[-idx]
mu <- colMeans(Xtr); sdv <- apply(Xtr,2,sd); Z <- scale(Xtr,mu,sdv); Zte <- scale(Xte,mu,sdv)
pc <- prcomp(Z, center=FALSE, scale.=FALSE); k <- 20; Ptr <- pc$x[,1:k]; Pte <- Zte %*% pc$rotation[,1:k]
set.seed(1); folds <- sample(rep(1:5, length.out=nrow(Ptr)))
cv_acc <- mean(sapply(1:5, function(f){ m <- svm(Ptr[folds!=f,], ytr[folds!=f], kernel="radial", cost=10, gamma=0.01)
  mean(predict(m, Ptr[folds==f,])==ytr[folds==f]) }))
svm_fit <- svm(Ptr, ytr, kernel="radial", cost=10, gamma=0.01)
mlp_fit <- nnet(Z, class.ind(ytr), size=16, softmax=TRUE, maxit=250, trace=FALSE, MaxNWts=3000)
cat(sprintf("%d simulated images, %d classes | PCA(%d)+SVM: CV acc %.3f, test acc %.3f | MLP: test acc %.3f\n",
    nrow(X), nlevels(yd), k, cv_acc, mean(predict(svm_fit,Pte)==yte), mean(levels(ytr)[max.col(predict(mlp_fit,Zte))]==yte)))



The simulated task is clean enough that both pipelines classify the held-out images essentially perfectly, and — the point of the chunk — the 5-fold cross-validated estimate on the training set agrees with the test score, as Supplement S1 requires. Hyper-parameters were chosen inside the training set, the test set was used once, both models were compared on the same split. The next chunk shows what the agreement does <em>not</em> protect against.



In [ ]:
# Domain shift: the same models applied to test images from a 'different scanner' (brighter, lower contrast, blurred).
other_scanner <- function(Xm, gain=0.7, offset=4, blur=TRUE) { out <- Xm*gain + offset
  if (blur) out <- t(apply(out, 1, function(img){ m <- matrix(img,8,8,byrow=TRUE); mb <- m
    for (i in 1:8) for (j in 1:8) mb[i,j] <- mean(m[max(1,i-1):min(8,i+1), max(1,j-1):min(8,j+1)]); as.vector(t(mb)) }))
  pmin(pmax(out,0),16) }
Xte_sh <- other_scanner(Xte); Zte_sh <- scale(Xte_sh,mu,sdv); Pte_sh <- Zte_sh %*% pc$rotation[,1:k]
acc_svm_sh <- mean(predict(svm_fit, Pte_sh)==yte); acc_mlp_sh <- mean(levels(ytr)[max.col(predict(mlp_fit, Zte_sh))]==yte)
cat(sprintf("accuracy  same scanner -> different scanner:  SVM %.3f -> %.3f | MLP %.3f -> %.3f\n",
    mean(predict(svm_fit,Pte)==yte), acc_svm_sh, mean(levels(ytr)[max.col(predict(mlp_fit,Zte))]==yte), acc_mlp_sh))
js <- sapply(levels(yte), function(cl) which(yte==cl)[1])                   # one example per class
par(mfrow=c(2,6), mar=c(1,1,1.6,0.3))
for (j in js) { image(matrix(Xte[j,],8,8,byrow=TRUE)[,8:1], col=gray.colors(16,1,0), axes=FALSE); title(paste("true", yte[j]), cex.main=0.8) }
for (j in js) { p <- predict(svm_fit, Pte_sh[j,,drop=FALSE])
  image(matrix(Xte_sh[j,],8,8,byrow=TRUE)[,8:1], col=gray.colors(16,1,0), axes=FALSE)
  title(paste("->", p), cex.main=0.8, col.main=ifelse(p==yte[j],GREEN,ORANGE)) }



<p class="caption"><b>Figure 6.</b> Six test images as acquired (top) and after a simulated change of scanner — lower contrast, a brightness offset, mild blur (bottom) — with the PCA + SVM prediction for each (green correct, orange wrong). The images remain legible to a person, yet accuracy falls from essentially perfect to about 0.15 for the SVM (chance for six classes) and about 0.28 for the MLP.</p>

The mechanism is typical: border pixels of the training images are almost always zero, so the standardiser learned a mean and variance near zero for them; the offset turns them into large standardised values no training image ever had, and both models extrapolate. The shift that hurts most is not the one most visible to a person (blur) but the one that moves the data furthest from the training distribution in the model's representation. The honest internal evaluation was honest about the wrong population, and the two models were hurt differently, so robustness is measured, not assumed. Remedies: external validation at the deployment site, harmonised acquisition, augmentation with the expected variations (Unit 8), monitoring after deployment (Section 5.6).

<div class="exercise"><h4>Exercise 3</h4><p>Vary the shift: set <code>blur=FALSE</code>; then <code>gain=1.0</code> with only the offset; then only the blur. Which component hurts each model most? Then re-fit both pipelines on training images augmented with their own shifted copies (<code>rbind(Xtr, other_scanner(Xtr))</code>) and re-test on both test sets. Does augmentation restore the shifted accuracy, and what does it cost on the original?</p></div>

## 4. Medical diagnostic systems (9.3)

### 4.1 A short history

Unit 1 described <strong>MYCIN</strong> (Stanford, 1972–1976), the rule-based antibiotic consultant. Its contemporary <strong>INTERNIST-1</strong> (Pittsburgh, from 1974) covered some 500 diseases and 3,550 manifestations, linked by an <em>evoking strength</em> and a <em>frequency</em>. In its 1982 evaluation on 19 clinicopathological-conference cases it performed comparably to the hospital clinicians and below the discussants, and its authors said plainly that it was not ready for clinical use: it could not reason about time, anatomy or causation, and could not explain itself.<sup class="c"><a href="#r12">[12]</a></sup> Its knowledge base became <strong>QMR</strong>; <strong>DXplain</strong> still generates differential diagnoses in teaching hospitals; and <strong>HELP</strong> (LDS Hospital, Salt Lake City) was the first hospital information system with embedded decision logic — alerts fired from laboratory and pharmacy data as they arrived — showing that the useful diagnostic system is a monitor that speaks when needed, not a consultant one must ask. These systems did not spread for the reasons Unit 1 gave for MYCIN — brittle hand-acquired knowledge, no place in the workflow, no owner of responsibility — and what survived was HELP's architecture and the records themselves.

### 4.2 Clinical decision support systems today

A <strong>clinical decision support system</strong> (CDSS) presents patient-specific information to a clinician at the point of care: **alerts and reminders** (interaction and dose checks — rule-based, plagued by <em>alert fatigue</em> with override rates above 90 %); **risk scores and prediction models**; **triage** of work-lists by predicted urgency; **diagnostic imaging AI** as second reader, pre-screen or, in one authorised case (Section 6), autonomous decision; and **order sets and documentation support**, since 2023 including language models drafting notes.

Osheroff and colleagues' implementer's guide distilled the <strong>five rights of CDSS</strong>: the right <em>information</em>, to the right <em>person</em> (who can act — not necessarily the physician), in the right <em>format</em>, through the right <em>channel</em>, at the right <em>time in the workflow</em> — the checklist for the deployment phase most projects skip. Sendak and colleagues' <em>model facts</em> label — one page stating what a model predicts, on whom it was validated, how it should and should not be used, whom to call — is the instrument for the right information in the right format.<sup class="c"><a href="#r31">[31]</a></sup> Rajkomar, Dean and Kohane make the complementary point: the hard problems at the bedside are the data pipeline, workflow integration and the evaluation of effect on care, not the learning algorithm.<sup class="c"><a href="#r30">[30]</a></sup>

### 4.3 Bayesian diagnostic reasoning and Fagan's nomogram

Every diagnostic system updates a belief with evidence (Unit 3). Clinical practice uses the <em>odds</em> form of Bayes' theorem, in which a test result acts by multiplication: with pre-test odds \(O_0 = p_0/(1-p_0)\) and the test's <strong>likelihood ratios</strong>,

<div class="eq">\[\text{LR}^{+} = \frac{\text{sensitivity}}{1 - \text{specificity}}, \qquad \text{LR}^{-} = \frac{1 - \text{sensitivity}}{\text{specificity}}, \qquad O_{\text{post}} = O_0 \times \text{LR}, \qquad p_{\text{post}} = \frac{O_{\text{post}}}{1 + O_{\text{post}}}\]
<span class="where">Sensitivity = specificity = 0.90 gives \(\text{LR}^{+} = 9\) and \(\text{LR}^{-} = 0.11\). On the logarithmic scale the update is additive, \(\log O_{\text{post}} = \log O_0 + \log \text{LR}\), which is why Fagan's nomogram is a straight line.</span></div>

Fagan's 1975 nomogram puts the three quantities on parallel logarithmic axes, so that a straight line from pre-test probability through likelihood ratio gives the post-test probability.<sup class="c"><a href="#r13">[13]</a></sup> It makes visible the central fact of diagnosis — <em>the same positive test means different things at different pre-test probabilities</em> — here for \(\text{LR}^{+} = 9\) in a screening setting (\(p_0 = 0.05\)) and a referral setting (\(p_0 = 0.37\), the prevalence in the Wisconsin data).



In [ ]:
# Fagan's nomogram: pre-test probability, likelihood ratio and post-test probability on parallel log-odds axes.
post_test <- function(p0, lr) { o <- p0/(1-p0)*lr; o/(1+o) }
logit <- function(p) log(p/(1-p)); sens <- 0.90; spec <- 0.90
lr_pos <- sens/(1-spec); lr_neg <- (1-sens)/spec
cases <- data.frame(p0=c(0.05,0.37,0.05,0.37), result=c("positive","positive","negative","negative"), LR=c(lr_pos,lr_pos,lr_neg,lr_neg))
cases$post <- mapply(post_test, cases$p0, cases$LR)
cat(sprintf("sensitivity %.2f, specificity %.2f -> LR+ = %.2f, LR- = %.3f\n", sens, spec, lr_pos, lr_neg))
print(transform(cases, post=round(post,3)), row.names=FALSE)
p_ticks <- c(.001,.002,.005,.01,.02,.05,.1,.2,.3,.5,.7,.8,.9,.95,.98,.99,.995,.998,.999)
lr_ticks <- c(.001,.002,.005,.01,.02,.05,.1,.2,.5,1,2,5,10,20,50,100,200,500,1000)
plot.new(); plot.window(xlim=c(-0.5,2.5), ylim=c(-7.6,9.0))
for (p in p_ticks) { text(-0.06, -logit(p), formatC(p*100,format="g"), adj=1, cex=0.6); text(2.06, logit(p), formatC(p*100,format="g"), adj=0, cex=0.6) }
for (lr in lr_ticks) text(1.05, log(lr)/2, formatC(lr,format="g"), adj=0, cex=0.6)
for (x in c(0,1,2)) segments(x,-7,x,7, lwd=1)
cols <- c(ORANGE,BLUE,ORANGE,BLUE)
for (i in 1:4) segments(0, -logit(cases$p0[i]), 2, logit(cases$post[i]), col=cols[i], lwd=1.6, lty=ifelse(cases$result[i]=="positive",1,3))
for (xt in list(c(0,"pre-test\nprobability (%)"), c(1,"likelihood\nratio"), c(2,"post-test\nprobability (%)"))) text(as.numeric(xt[1]), 8.4, xt[2], cex=0.7, font=2)
legend("bottom", sprintf("pre %.0f%%, %s (LR %.2f) -> post %.1f%%", cases$p0*100, cases$result, cases$LR, cases$post*100),
       col=cols, lty=ifelse(cases$result=="positive",1,3), lwd=1.6, bty="n", cex=0.65)
title("Fagan's nomogram for a test with sensitivity = specificity = 0.90", cex.main=0.9)



<p class="caption"><b>Figure 7.</b> Fagan's nomogram computed in code, on log-odds scales so that Bayes' theorem is a straight line. The same positive result (solid, LR⁺ = 9) takes a screening patient from 5 % to about 32 % and a referred patient from 37 % to about 84 %; the same negative result (dotted, LR⁻ = 0.11) takes them to about 0.6 % and 6 %. A "90 % accurate" test leaves two of three positive screening patients disease-free.</p>

<div class="box key"><h4>What this means for a learning system</h4><p>A classifier's output is a post-test probability <em>at the prevalence of its training set</em>. Deployed where prevalence differs, its probabilities are wrong by exactly the prior-odds ratio even if its likelihood ratios are unchanged. Hence Section 4.4 reports the prevalence of its evaluation set, calibration must be re-checked at every site, and a positive predictive value quoted without a prevalence is meaningless.</p></div>

### 4.4 The capstone project: a complete and honest diagnostic model

The rest of this section builds a diagnostic model for the Wisconsin breast-cancer data as a model intended for patients should be built, following Supplement S1 and the standards of Section 4.6:

1. **Problem and data.** Positive class = malignant. Stratified 75/25 split; the test set is touched once, at the end.
2. **Candidate models.** RBF support vector machine, random forest, multilayer perceptron (Units 3, 4, 6), each standardised so that no scaling statistic leaks across folds.
3. **Nested cross-validation**: inner 3-fold hyper-parameter search, outer 5-fold estimate; the family with the highest outer AUC is selected.
4. **Calibration** by Platt scaling, fitted on cross-validated training predictions.
5. **Evaluation on the test set**: ROC-AUC, reliability curve and Brier score, decision curve.
6. **Threshold** from the relative cost of a missed cancer and an unnecessary biopsy, chosen on cross-validated training predictions.
7. **Subgroup check** and **model card**.



In [ ]:
# Step 1-2: stratified split, three candidate families as standardised fits, and search grids.
auc_of <- function(truth, prob) as.numeric(auc(truth, prob, levels=c("ben","mal"), direction="<", quiet=TRUE))
std_fit <- function(Xt) list(mu=colMeans(Xt), sd=apply(Xt,2,sd)); std_ap <- function(s,Xt) scale(Xt, s$mu, s$sd)
make_folds <- function(y,k,seed){ set.seed(seed); f<-integer(length(y)); for(c in levels(y)){i<-which(y==c); f[i]<-sample(rep(1:k,length.out=length(i)))}; f }
fit_predict <- function(fam, par, Xt, yt, Xn){ s<-std_fit(Xt); Zt<-std_ap(s,Xt); Zn<-std_ap(s,Xn)
  if (fam=="svm")      attr(predict(svm(Zt,yt,kernel="radial",cost=par$C,gamma=par$gamma,probability=TRUE), Zn, probability=TRUE),"probabilities")[,"mal"]
  else if (fam=="rf")  predict(randomForest(Zt,yt,ntree=150,mtry=par$mtry,nodesize=par$nodesize), Zn, type="prob")[,"mal"]
  else                 predict(nnet(Zt,class.ind(yt),size=par$size,decay=par$decay,maxit=400,trace=FALSE,MaxNWts=3000), Zn)[,"mal"] }
grids <- list(svm=expand.grid(C=c(0.3,1,3,10), gamma=c(0.01,0.03,0.1)),
              rf=expand.grid(mtry=c(2,3), nodesize=c(1,3)), mlp=expand.grid(size=c(4,8), decay=c(0.01,0.1)))
fam_label <- c(svm="SVM, RBF (Unit 3)", rf="Random forest (Unit 4)", mlp="MLP (Unit 6)")
tune <- function(fam, Xt, yt, inner_k=3, seed=0){ g<-grids[[fam]]; f<-make_folds(yt,inner_k,seed); aucs<-numeric(nrow(g))
  for (gi in 1:nrow(g)) aucs[gi] <- mean(sapply(1:inner_k, function(k) auc_of(yt[f==k], fit_predict(fam, as.list(g[gi,]), Xt[f!=k,], yt[f!=k], Xt[f==k,]))))
  list(best=as.list(g[which.max(aucs),]), auc=max(aucs)) }
X <- as.matrix(bc[,1:9]); y <- factor(ifelse(bc$Class=="malignant","mal","ben"), levels=c("ben","mal"))
set.seed(0); tr <- unlist(lapply(levels(y), function(c){ i<-which(y==c); sample(i, round(0.75*length(i))) }))
Xtr <- X[tr,]; ytr <- y[tr]; Xte <- X[-tr,]; yte <- y[-tr]
cat(sprintf("training set: %d cases (%.1f%% malignant) | test set: %d cases (%.1f%% malignant), locked away\n",
    length(ytr), 100*mean(ytr=="mal"), length(yte), 100*mean(yte=="mal")))
cat("hyper-parameter combinations per family:", paste(names(grids), sapply(grids, nrow), sep="="), "\n")



The 683 aspirates split into 512 training and 171 test cases, each 35 % malignant by stratification. (The scikit-learn copy had 569 cases; the mlbench copy is larger, so these counts differ from the Python edition while the problem is identical.) The test set is now set aside and is not consulted until step 5.



In [ ]:
# Step 3: nested cross-validation (inner 3-fold tuning, outer 5-fold estimate) of the three families.
outer <- make_folds(ytr, 5, 0); nested <- data.frame()
for (fam in c("svm","rf","mlp")) { set.seed(0); aucs <- accs <- numeric(5); first_par <- NULL
  for (k in 1:5) { best <- tune(fam, Xtr[outer!=k,], ytr[outer!=k], 3, 0); if (k==1) first_par <- best$best
    pr <- fit_predict(fam, best$best, Xtr[outer!=k,], ytr[outer!=k], Xtr[outer==k,])
    aucs[k] <- auc_of(ytr[outer==k], pr); accs[k] <- mean((pr>=0.5)==(ytr[outer==k]=="mal")) }
  nested <- rbind(nested, data.frame(family=fam_label[fam], outer_AUC=mean(aucs), AUC_sd=sd(aucs), outer_acc=mean(accs),
    best_fold1=paste(names(first_par), unlist(first_par), sep="=", collapse=", "))) }
chosen_fam <- c("svm","rf","mlp")[which.max(nested$outer_AUC)]
cat("selected family (highest outer AUC):", fam_label[chosen_fam], "\n")
nested2 <- nested; nested2[,2:4] <- round(nested2[,2:4], 4); print(nested2, row.names=FALSE)



All three families reach an outer-fold AUC above 0.98 and differ by a few thousandths — less than the fold-to-fold standard deviation — so the table gives no strong grounds for preference; the code selects the highest mean, in this run the support vector machine, and the selection is itself a choice made on training data. The multilayer perceptron is fitted with <code>nnet</code>, whose single-hidden-layer network and <code>lda</code> companion are documented in Venables and Ripley's <em>Modern Applied Statistics with S</em>.<sup class="c"><a href="#r41">[41]</a></sup> Unit 3 made the same comparison without a locked-away test set; only the present numbers estimate performance on new cases, because the test set has not yet been used.



In [ ]:
# Step 4-5a: refit the selected family on all training data, calibrate by Platt scaling, evaluate once on the test set.
best_overall <- tune(chosen_fam, Xtr, ytr, 3, 0)
oof <- numeric(length(ytr)); ocv <- make_folds(ytr,5,0)                    # out-of-fold training scores (cross_val_predict)
for (k in 1:5) oof[ocv==k] <- fit_predict(chosen_fam, best_overall$best, Xtr[ocv!=k,], ytr[ocv!=k], Xtr[ocv==k,])
clamp <- function(p) pmin(pmax(p,1e-6),1-1e-6)
platt <- glm((ytr=="mal") ~ qlogis(clamp(oof)), family=binomial); co <- coef(platt)   # Platt sigmoid on the logit of the score
raw_te <- fit_predict(chosen_fam, best_overall$best, Xtr, ytr, Xte)
p_te <- plogis(co[1] + co[2]*qlogis(clamp(raw_te)))
auc_te <- auc_of(yte, p_te); brier <- mean((p_te - (yte=="mal"))^2); prev <- mean(yte=="mal")
cm <- table(truth=yte, pred=factor(ifelse(p_te>=0.5,"mal","ben"), levels=c("ben","mal")))
tn<-cm[1,1]; fp<-cm[1,2]; fn<-cm[2,1]; tp<-cm[2,2]
cat(sprintf("tuned parameters: %s\n", paste(names(best_overall$best), unlist(best_overall$best), sep="=", collapse=", ")))
cat(sprintf("test set (%d cases): ROC-AUC %.3f | Brier %.3f (%.3f for a constant prediction at prevalence)\n", length(yte), auc_te, brier, prev*(1-prev)))
cat(sprintf("at threshold 0.5: TP %d, FN %d, FP %d, TN %d -> sensitivity %.3f, specificity %.3f, PPV %.3f\n",
    tp, fn, fp, tn, tp/(tp+fn), tn/(tn+fp), tp/(tp+fp)))



Fitted once on 512 training cases and evaluated once on 171 test cases, the model reaches a test AUC of about 0.997 and a Brier score of about 0.013 (against 0.228 for predicting the prevalence for everyone); at threshold 0.5 it misses 2 of 60 malignant cases and raises 1 false alarm among 111 benign ones.<sup class="c"><a href="#r38">[38]</a>,<a href="#r39">[39]</a>,<a href="#r40">[40]</a></sup> These are the numbers a paper reports. The next chunks ask what a paper usually does not: are the probabilities calibrated, is the model useful at the thresholds clinicians use, and is 0.5 the right threshold?



In [ ]:
# Step 5b: calibration - do predicted probabilities match observed frequencies? Compared with the uncalibrated random forest.
p_rf <- fit_predict("rf", tune("rf", Xtr, ytr, 3, 0)$best, Xtr, ytr, Xte); brier_rf <- mean((p_rf-(yte=="mal"))^2)
reliability <- function(yv, p, nb=6){ q <- unique(quantile(p, seq(0,1,length.out=nb+1))); q[1]<- -Inf; q[length(q)]<-Inf
  b <- cut(p, q, include.lowest=TRUE); na.omit(data.frame(mp=tapply(p,b,mean), obs=tapply(as.integer(yv=="mal"),b,mean))) }
par(mfrow=c(1,2), mar=c(4,4,2,1))
rc <- reliability(yte, p_te); rr <- reliability(yte, p_rf)
plot(rc$mp, rc$obs, type="o", pch=19, col=BLUE, xlim=0:1, ylim=0:1, xlab="mean predicted probability (bin)",
     ylab="observed fraction malignant", main="Reliability curve (test set, 6 quantile bins)", cex.main=0.85)
lines(rr$mp, rr$obs, type="o", pch=19, col=ORANGE); abline(0,1,lty=2)
legend("topleft", c(sprintf("calibrated SVM (Brier %.3f)",brier), sprintf("random forest (Brier %.3f)",brier_rf), "perfect"),
       col=c(BLUE,ORANGE,"black"), lty=c(1,1,2), pch=c(19,19,NA), bty="n", cex=0.7)
hist(p_te[yte=="ben"], breaks=seq(0,1,0.04), col=adjustcolor(BLUE,0.7), border="white", xlab="predicted probability of malignancy",
     ylab="test cases", main="Predicted probabilities by true class", cex.main=0.85)
hist(p_te[yte=="mal"], breaks=seq(0,1,0.04), col=adjustcolor(ORANGE,0.7), border="white", add=TRUE)
legend("top", c("benign","malignant"), fill=c(adjustcolor(BLUE,0.7),adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figure 8.</b> Left: reliability curves on the test set — cases binned by predicted probability, mean prediction against observed fraction malignant; a calibrated model lies on the diagonal. Right: predicted probabilities of the calibrated model by true class; most cases sit near 0 or 1. The Brier score (legend) is the mean squared difference between predicted probability and outcome.</p>

Discrimination and calibration are different properties: a model can rank cases perfectly yet report probabilities that are systematically too timid or too bold. Platt scaling fits a logistic regression from the score to the outcome on cross-validated training predictions and corrects such distortion. On these nine clean features the uncalibrated random forest is already close to the diagonal (its Brier score is only marginally above the calibrated model's 0.013), so the correction is small here — a caveat the Python edition's thirty-feature data made larger. Calibration still matters because the threshold of step 6 and the net benefit of step 5 live <em>on the probability scale</em>.



In [ ]:
# Step 5c: decision-curve analysis (Vickers & Elkin 2006) in base R - net benefit against the threshold probability.
net_benefit <- function(yv, p, th) { yy <- as.integer(yv=="mal"); n <- length(yy)
  sapply(th, function(t){ pr <- p>=t; sum(pr & yy==1)/n - sum(pr & yy==0)/n * t/(1-t) }) }
th <- seq(0.01,0.79,0.01); nb_model <- net_benefit(yte,p_te,th); nb_rf <- net_benefit(yte,p_rf,th)
nb_all <- prev - (1-prev)*th/(1-th)
dt <- c(0.05,0.10,0.20,0.33,0.50)
dca <- data.frame(threshold=dt, model=net_benefit(yte,p_te,dt), biopsy_all=prev-(1-prev)*dt/(1-dt), biopsy_none=0)
cat("net benefit (true positives per 100 patients, net of harm from false positives):\n"); print(round(dca*100,1), row.names=FALSE)
plot(th, nb_model, type="l", col=BLUE, lwd=2, ylim=c(-0.05, prev+0.05), xlab="threshold probability (where a clinician would act)",
     ylab="net benefit", main="Decision curve on the test set", cex.main=0.9)
lines(th, nb_rf, col=ORANGE, lwd=1.2); lines(th, nb_all, col=GREY, lwd=1.2, lty=2); abline(h=0, lwd=1)
legend("topright", c("calibrated SVM","random forest","biopsy everyone","biopsy no one"),
       col=c(BLUE,ORANGE,GREY,"black"), lty=c(1,1,2,1), lwd=c(2,1.2,1.2,1), bty="n", cex=0.75)



<p class="caption"><b>Figure 9.</b> Decision curves on the test set. The horizontal axis is the <em>threshold probability</em> \(p_t\) at which a clinician or patient would biopsy — how a missed cancer is weighed against an unnecessary procedure; the vertical axis is net benefit. "Biopsy everyone" (dashed) and "biopsy no one" (zero line) bound the region in which a model is useful; the calibrated model lies above both across the plausible range.</p>

Vickers and Elkin's decision-curve analysis answers what AUC and calibration do not: <em>would deciding with this model lead to better decisions than not using it?</em><sup class="c"><a href="#r14">[14]</a></sup> Its one assumption is that a clinician who acts at probability \(p_t\) values a true positive at \((1-p_t)/p_t\) false positives — someone who biopsies at 10 % accepts nine unnecessary biopsies per cancer found — so that

<div class="eq">\[\text{NB}(p_t) = \frac{\text{TP}(p_t)}{n} - \frac{\text{FP}(p_t)}{n} \cdot \frac{p_t}{1 - p_t}\]
<span class="where">in true positives per patient. "Biopsy everyone" gives \(\text{NB} = \pi - (1-\pi)\,p_t/(1-p_t)\) with \(\pi\) the prevalence; "biopsy no one" gives 0. A model is useful at \(p_t\) only if its curve lies above both.</span></div>

The table above the figure reads the curve at five thresholds: at \(p_t = 0.10\) the model yields about 35 true positives per 100 patients net of harm against about 28 for biopsying everyone. Near \(p_t \to 0\) every strategy converges to "biopsy everyone": a clinician who will biopsy at any probability needs no model.

### 4.5 Choosing the threshold by clinical cost

The default threshold of 0.5 treats a missed cancer and an unnecessary biopsy as equally costly. If a missed malignancy is \(c\) times worse, the expected cost of a threshold is \(c \cdot \text{FN} + \text{FP}\) and the decision-theoretic optimum (Unit 3) is to act when \(p > 1/(1+c)\). The chunk chooses the threshold empirically from <em>cross-validated training predictions</em> and reports the test-set confusion matrix and likelihood ratios at that threshold.



In [ ]:
# Step 6: threshold by clinical cost (FN = c x FP), chosen on cross-validated TRAINING predictions, applied once to the test set.
expected_cost <- function(yv,p,t,c){ pr<-p>=t; c*sum(!pr & yv=="mal") + sum(pr & yv=="ben") }
grid_t <- seq(0.02,0.94,0.01); thr_rows <- data.frame()
for (cc in c(1,3,5,10)) { tstar <- grid_t[which.min(sapply(grid_t, function(t) expected_cost(ytr, oof, t, cc)))]
  cmx <- table(yte, factor(ifelse(p_te>=tstar,"mal","ben"), levels=c("ben","mal")))
  tn<-cmx[1,1];fp<-cmx[1,2];fn<-cmx[2,1];tp<-cmx[2,2]; se<-tp/(tp+fn); sp<-tn/(tn+fp)
  thr_rows <- rbind(thr_rows, data.frame(cost_ratio=cc, theoretical=1/(1+cc), threshold=tstar, test_FN=fn, test_FP=fp,
    sensitivity=se, specificity=sp, LRpos=ifelse(sp<1, se/(1-sp), Inf), LRneg=(1-se)/sp)) }
T_STAR <- thr_rows$threshold[thr_rows$cost_ratio==5]
cat(sprintf("operating threshold adopted (c = 5): %.2f\n", T_STAR))
tr2 <- thr_rows; num <- c("theoretical","threshold","sensitivity","specificity","LRpos","LRneg"); tr2[,num] <- round(tr2[,num],3)
print(tr2, row.names=FALSE)



As the cost ratio rises the threshold should fall and sensitivity rise. Here the calibrated probabilities are so sharply separated — most cases sit within a hundredth of 0 or 1 — that the cost surface is flat and all four cost ratios select the same operating threshold, about 0.14, which already drives test-set false negatives to zero at the price of two false positives. The empirical threshold therefore tracks \(1/(1+c)\) only loosely, a stronger version of the same caveat the Python edition noted. We adopt \(c = 5\). The likelihood ratios at that threshold read directly into Figure 7: the post-test probability after a positive or negative output depends on the prevalence where the model is used.

The cost ratio is not the engineer's to choose alone: it encodes a clinical and ethical judgement that belongs to clinicians, patients and the institution. The engineer's obligation is to expose the choice and show its consequences, as the table and the decision curve do.

### 4.6 Subgroup check, model card and reporting standards

A model good on average may be bad on a subgroup — an age range, a scanner, a hospital. The Wisconsin data carry no demographic variables, so the chunk simulates the simplest check: test cases assigned at random to two "hospitals", where performance should be equal by construction. Section 5.2 repeats it with a subgroup variable <em>correlated</em> with the outcome.



In [ ]:
# Step 7a: performance by subgroup. Here the subgroup is a random 'hospital' label, so differences are sampling noise.
subgroup_report <- function(yv,p,group,t) do.call(rbind, lapply(sort(unique(group)), function(gg){ m<-group==gg
  cmx <- table(factor(yv[m],levels=c("ben","mal")), factor(ifelse(p[m]>=t,"mal","ben"),levels=c("ben","mal")))
  tn<-cmx[1,1];fp<-cmx[1,2];fn<-cmx[2,1];tp<-cmx[2,2]
  data.frame(group=gg, n=sum(m), prevalence=mean(yv[m]=="mal"), AUC=auc_of(yv[m],p[m]),
             sensitivity=tp/(tp+fn), specificity=tn/(tn+fp), flagged=mean(p[m]>=t)) }))
set.seed(0); hospital <- sample(c("Hospital A","Hospital B"), length(yte), replace=TRUE, prob=c(0.6,0.4))
sg <- subgroup_report(yte, p_te, hospital, T_STAR); sg[,3:7] <- round(sg[,3:7],3); print(sg, row.names=FALSE)



The two hospitals have the same sensitivity and differ by about half a hundredth in specificity, with roughly 65–105 cases each. First lesson: with \(n \approx 65\)–105 per group one additional false negative moves sensitivity by two to four points, so such differences are compatible with equal performance, and confidence intervals, not point estimates, should be reported. Second (Section 2.6): of ten subgroups examined, the worst will look bad by chance, and a subgroup finding must be confirmed on new data before it is acted on.

The <strong>model card</strong> of Mitchell and colleagues is a structured document that states, in fixed fields, what a model is and is for, on what data it was trained and evaluated, how it performs overall and by subgroup, and its limitations and ethical considerations.<sup class="c"><a href="#r15">[15]</a></sup> The chunk assembles one from the quantities computed above, so that every field is traceable to a chunk of this notebook.



In [ ]:
# Step 7b: a model card assembled from the computed quantities (Mitchell et al. 2019).
r5 <- thr_rows[thr_rows$cost_ratio==5,]
card <- data.frame(field=c("Model details","Intended use","Out-of-scope use","Training data","Evaluation data",
  "Discrimination","Calibration","Operating point","Performance at operating point","Clinical utility",
  "Subgroup performance","Ethical considerations","Caveats and limitations"),
  content=c(
  sprintf("%s; standardise -> tuned estimator (%s) -> Platt (sigmoid) calibration on 5-fold training predictions. R %s, e1071/randomForest/nnet.", fam_label[chosen_fam], paste(names(best_overall$best),unlist(best_overall$best),sep="=",collapse=", "), getRversion()),
  "Probability that a fine-needle aspirate is malignant, from nine nuclear-morphology scores, to support (not replace) the decision to biopsy in a referral setting.",
  "Screening populations (prevalence far below 35%), images from other laboratories or feature software, any use without a pathologist in the loop.",
  sprintf("%d aspirates (Wisconsin, mlbench copy), %.1f%% malignant.", length(ytr), 100*mean(ytr=="mal")),
  sprintf("%d held-out aspirates from the same source (internal validation only), %.1f%% malignant. No external validation.", length(yte), 100*mean(yte=="mal")),
  sprintf("ROC-AUC %.3f on the test set; nested-CV AUC %.3f +/- %.3f.", auc_te, nested$outer_AUC[which.max(nested$outer_AUC)], nested$AUC_sd[which.max(nested$outer_AUC)]),
  sprintf("Brier score %.3f; reliability curve close to the diagonal (Figure 8).", brier),
  sprintf("Threshold %.2f, chosen on cross-validated training predictions for a cost ratio of 5.", T_STAR),
  sprintf("Sensitivity %.3f, specificity %.3f; %d false negatives and %d false positives among %d test cases.", r5$sensitivity, r5$specificity, r5$test_FN, r5$test_FP, length(yte)),
  "Net benefit above 'biopsy all' and 'biopsy none' for threshold probabilities 0.05-0.50 (Figure 9).",
  "Equal within sampling error across two random hospitals; no demographic variables, so fairness by age/ethnicity/socioeconomic group is UNTESTED.",
  "Label is the biopsy result; features depend on one laboratory's imaging. Any deployment requires local validation, monitoring and informed patients.",
  "Single-site, decades-old data; small test set (wide intervals); probabilities valid only at training prevalence; not evaluated prospectively."), stringsAsFactors=FALSE)
knitr::kable(card, col.names=c("Field","Content"))



Every row of the card is a number computed above or a statement of what was <em>not</em> done — internal validation only, fairness untested, prevalence of use restricted, no prospective evaluation — and the second kind matters more: a reader knows what evidence is missing before deciding whether to use the model.

**Reporting standards.** <strong>TRIPOD+AI</strong> (2024) governs the report of a prediction model's development and validation — data, outcome, sample size, model building, every performance measure with its uncertainty, fairness, availability of code and data: the model card as a manuscript checklist.<sup class="c"><a href="#r16">[16]</a></sup> <strong>DECIDE-AI</strong> (2022) governs the <em>early clinical evaluation</em> of a decision-support system and asks for what a development paper cannot contain: how users interacted with it, how often they overrode it, what errors arose, how safety was monitored.<sup class="c"><a href="#r18">[18]</a></sup> <strong>CONSORT-AI</strong> and <strong>SPIRIT-AI</strong> (2020) extend the standards for randomised trials and protocols to interventions with an AI component.<sup class="c"><a href="#r17">[17]</a></sup> The three answer the three questions of Section 3.3: does the model predict, can clinicians use it, does using it improve care?

### 4.7 Why most prediction models never reach practice

Thousands of clinical prediction models are published every year and very few are used. COVID-19 gave the clearest demonstration: Wynants and colleagues' living systematic review assessed every published diagnostic or prognostic model — 232 by mid-2020, several hundred since — with the PROBAST risk-of-bias tool. Almost all were at high or unclear risk of bias, for reasons that catalogue this course's warnings — unrepresentative controls, exclusion of patients who had not yet had the outcome, overfitting from few events, reporting too vague to permit validation, performance estimated on development data — and none was recommended for clinical use.<sup class="c"><a href="#r19">[19]</a></sup> The failure was not the algorithms — mostly logistic regressions and tree ensembles — but the <em>process</em>: evaluation, leakage, external validation. Topol's review reached the same conclusion from the other direction: of the many systems matching specialists retrospectively, a handful have been evaluated prospectively and fewer in trials measuring patient outcomes.<sup class="c"><a href="#r28">[28]</a></sup> The capstone model, single-site and internally validated, is a development-stage model, and its card says so.

<div class="exercise"><h4>Exercise 4</h4><p>Force the MLP as the selected family (<code>chosen_fam <- "mlp"</code> before step 4) and compare calibration curve, decision curve and operating threshold with the SVM's. Then introduce a deliberate leak — standardise on all 683 cases before the split — and observe whether any number in the model card changes. Why is this particular leak small here, and when would it be large?</p></div>

## 5. Professional ethics in working with data (9.4)

Unit 1 introduced the ethics of health data through re-identification, dataset shift, label bias and the regulatory frame. This section builds on it in six parts: the data life cycle; fairness defined precisely enough to be measured; explainability and its limits; privacy beyond de-identification; governance and law; the closing checklist. The register is that of Char, Shah and Magnus: the ethical questions are not separate from the engineering ones, because the choices of label, data, threshold and deployment <em>are</em> the ethical choices.<sup class="c"><a href="#r32">[32]</a></sup>

### 5.1 The data life cycle and the engineer's duties

Data about patients pass through a sequence of states, and the engineer who builds a learning system is responsible at every one.



In [ ]:
# The data life cycle with the engineer's duty at each stage (schematic).
stg <- list(c("Collection","lawful basis; minimisation;\nfit for the stated purpose"),
  c("Consent","informed, specific, revocable;\nethics-committee approval"),
  c("Storage","encryption; pseudonymisation;\nkey kept separately"),
  c("Access","role-based; logged;\nneed-to-know; data-use agreement"),
  c("Analysis","S1 methodology; fairness audit;\nno re-identification attempts"),
  c("Publication","reporting standards; model card;\nno individual-level disclosure"),
  c("Retention /\ndeletion","retain only as long as justified;\nsecure deletion; right to erasure"))
plot.new(); plot.window(xlim=c(0,10), ylim=c(0,4.7)); w <- 9.4/length(stg)
for (i in seq_along(stg)) { x <- 0.3 + (i-1)*w
  rect(x, 2.6, x+w*0.86, 3.5, col=adjustcolor(BLUE,0.15), border=BLUE)
  text(x+w*0.43, 3.05, stg[[i]][1], font=2, col=BLUE, cex=0.72)
  yd <- ifelse(i%%2==1, 1.85, 1.0); segments(x+w*0.43, 2.6, x+w*0.43, yd+0.28, col=GREY, lty=3)
  text(x+w*0.43, yd, stg[[i]][2], cex=0.6)
  if (i < length(stg)) arrows(x+w*0.86, 3.05, x+w, 3.05, length=0.06, lwd=1) }
text(5, 0.25, "the cycle closes: data retained for a new purpose require a new basis and, usually, new consent", col=GREY, font=3, cex=0.7)
title("The data life cycle and the engineer's duty at each stage", cex.main=0.95)



<p class="caption"><b>Figure 10.</b> Seven stages of the life of a health dataset with the engineer's duty at each — usually the same person who requested the extract, holds the key to the pseudonyms and decides what is published.</p>

**Collection and purpose.** Only for a stated purpose and only to the extent needed (<em>minimisation</em>); the GDPR and Chile's Law 21.719 make "collect everything" unlawful without a basis, and data collected for care are not thereby available for research. **Consent and ethics review.** Informed, specific, revocable consent or a justified waiver, plus approval by a research ethics committee (in Chile the <em>Comité Ético Científico</em>, Law 20.120). **Storage and access.** Pseudonymisation with the key held separately, encryption, logged role-based access, a data-use agreement — the engineer who copies an extract to a laptop has broken all four. **Analysis.** The methodology of S1 is an ethical requirement — a model overstated by leakage harms the patients on whom it fails — with a fairness audit and no re-identification attempts without authorisation. **Publication.** The standards of Section 4.6; no individual disclosed; code and, where lawful, data shared. **Retention and deletion.** Only as long as the purpose requires, deleted securely, the right to erasure honoured — and a trained model may itself retain information about individuals (Section 5.4).

### 5.2 Fairness: definitions, the impossibility result and an audit

"Fair" has several precise meanings for a classifier, and they conflict. With outcome \(Y\), decision \(\hat Y\), predicted probability \(\hat p\) and a sensitive attribute \(A\) with groups \(a, b\):

<div class="eq">\[\begin{aligned}
\text{Demographic parity:}\quad & P(\hat Y = 1 \mid A = a) = P(\hat Y = 1 \mid A = b) \\
\text{Equalised odds:}\quad & P(\hat Y = 1 \mid Y = y, A = a) = P(\hat Y = 1 \mid Y = y, A = b) \quad \text{for } y \in \{0, 1\} \\
\text{Calibration within groups:}\quad & P(Y = 1 \mid \hat p = p, A = a) = P(Y = 1 \mid \hat p = p, A = b) = p \quad \text{for all } p
\end{aligned}\]
<span class="where">The same fraction of each group flagged / equal sensitivity and false-positive rate / a predicted 0.3 means 30 % in every group.</span></div>

Demographic parity is appropriate when base rates <em>should</em> be equal and any difference is label bias — the Obermeyer case of Unit 1.<sup class="c"><a href="#r29">[29]</a></sup> It is inappropriate when base rates genuinely differ: breast-cancer incidence rises with age, and flagging equal fractions of 30- and 65-year-olds would miss cancers in the old or over-treat the young. Equalised odds and calibration are then the criteria — and Chouldechova (analysing the COMPAS recidivism score) and Kleinberg, Mullainathan and Raghavan (in a general theorem) proved that <strong>when base rates differ between groups and the classifier is imperfect, calibration within groups and equal false-positive and false-negative rates cannot all hold at once</strong>.<sup class="c"><a href="#r20">[20]</a>,<a href="#r21">[21]</a></sup> One must <em>choose</em> which fairness to satisfy, and the choice is a value judgement about the harms of the application. The chunk audits the capstone model by a synthetic age group in which older patients are more often malignant, as in reality.



In [ ]:
# Fairness audit by a synthetic age group whose base rate of malignancy differs (older patients: more malignant).
set.seed(1); older <- ifelse(yte=="mal", runif(length(yte))<0.70, runif(length(yte))<0.35)   # P(older|mal)=.70, P(older|ben)=.35
age_group <- ifelse(older, ">= 55 years", "< 55 years")
au <- subgroup_report(yte, p_te, age_group, T_STAR)
au$false_positive_rate <- 1 - au$specificity
au$mean_predicted_p <- sapply(au$group, function(gg) mean(p_te[age_group==gg]))
au$observed_rate <- au$prevalence
au <- au[, c("group","n","prevalence","flagged","sensitivity","false_positive_rate","mean_predicted_p","observed_rate","AUC")]
cat("demographic parity -> 'flagged'; equalised odds -> sensitivity & FPR; calibration -> mean predicted vs observed\n")
num <- setdiff(names(au), c("group","n")); au[,num] <- round(au[,num],3); print(au, row.names=FALSE)



<em>Demographic parity</em> fails by construction — the older group is flagged far more often (about 0.51 against 0.22) because it has far more malignancy — and that is correct behaviour. <em>Equalised odds</em> holds approximately (similar sensitivity, both at the ceiling, and false-positive rates within the sampling error of groups this size), and so does <em>calibration within groups</em> (mean predicted probability tracks observed malignancy, about 0.49 against an observed 0.49 in the older group and 0.20 against 0.21 in the younger). The audit passes on the criteria that matter here and fails on the one that does not, and the model card must say which is which. Had age been correlated with the <em>features</em> as well — older patients on an older scanner — the model could have learned it as a shortcut and the error rates would have diverged.

<div class="box warn"><h4>Fairness is not a property of the model alone</h4><p>A model fair on its test set can produce unfair outcomes when deployed — if one group has less access to the clinic, or clinicians override it differently for different patients. Fairness is audited at the level of the system and its outcomes over time (Section 5.6). Conversely, "we did not use ethnicity as a feature" guarantees nothing: Unit 1 showed bias entering through the label alone.</p></div>

### 5.3 Explainability and transparency

A clinician asked to act on a model's output needs to know <em>why</em> — to check it against the patient, to catch failures, to carry the responsibility the model cannot. Explanations are **global** (what the model depends on), **local** (why one patient received one output) or **intrinsic** — logistic regression, a shallow tree, a scoring system, whose computation is the explanation. The chunk gives two global ones, implemented in base R: <strong>permutation importance</strong>, the loss of test AUC when one feature is shuffled, and <strong>partial dependence</strong>, the average prediction as one feature varies.



In [ ]:
# Global explanations: permutation importance (drop in test AUC when a feature is shuffled) and partial dependence.
sc <- std_fit(Xtr)
svm_final <- svm(std_ap(sc,Xtr), ytr, kernel="radial", cost=best_overall$best$C, gamma=best_overall$best$gamma, probability=TRUE)
predict_final <- function(Xn){ raw <- attr(predict(svm_final, std_ap(sc,Xn), probability=TRUE),"probabilities")[,"mal"]
  plogis(co[1] + co[2]*qlogis(clamp(raw))) }
set.seed(0); base_auc <- auc_of(yte, predict_final(Xte))
imp <- sort(sapply(colnames(Xte), function(f) mean(replicate(10, { Xp<-Xte; Xp[,f]<-sample(Xp[,f]); base_auc - auc_of(yte, predict_final(Xp)) }))), decreasing=TRUE)
cat("permutation importance (mean drop in test AUC over 10 shuffles), top 6:\n"); print(round(head(imp,6),4))
cat(sprintf("features whose shuffling changes the AUC by < 0.001: %d of %d\n", sum(abs(imp)<0.001), length(imp)))
top2 <- names(imp)[1:2]
par(mfrow=c(1,3), mar=c(4,7,2,1))
barplot(rev(head(imp,9)), horiz=TRUE, las=1, col=BLUE, cex.names=0.7, main="Permutation importance", xlab="drop in test AUC", cex.main=0.85)
par(mar=c(4,4,2,1))
for (f in top2) { gx <- seq(min(Xte[,f]), max(Xte[,f]), length.out=20)
  pd <- sapply(gx, function(v){ Xp<-Xte; Xp[,f]<-v; mean(predict_final(Xp)) })
  plot(gx, pd, type="l", col=ORANGE, lwd=2, xlab=f, ylab="mean predicted p(malignant)", main=paste("partial dependence:", f), cex.main=0.8) }



<p class="caption"><b>Figure 11.</b> Left: permutation importance of the most influential features — the mean drop in test AUC when the column is shuffled. "Bare nuclei" and "clump thickness" head the list; all drops are below 0.01 because the AUC is near its ceiling and the features are redundant, and 7 of 9 can be shuffled with essentially no effect. Right: partial dependence on the two leading features — both rise with the score, as pathology predicts; the range is narrow because the curve averages over the other features.</p>

Both tools fail with correlated features — a shuffled feature's correlates stay intact, so a duplicated feature looks unimportant; partial dependence evaluates the model at combinations that never occur — and both explain the <em>model</em>, not the <em>disease</em>.

**A local explanation and the interpretable alternative.** For a linear model one patient's output decomposes exactly: the logit is \(\beta_0 + \sum_j \beta_j z_j\), so each standardised feature contributes \(\beta_j z_j\). For a black box, LIME and SHAP fit a local surrogate or compute game-theoretic attributions — approximations. Rudin argues that in high-stakes decisions the approximation is the problem: it can be wrong exactly where it matters, and on tabular data an interpretable model usually performs as well.<sup class="c"><a href="#r22">[22]</a></sup> The chunk tests the claim: it fits the logistic regression of Unit 2, compares its AUC with the black box, and explains one borderline patient exactly.



In [ ]:
# An interpretable model beside the black box, and an exact local explanation of one borderline test patient.
Ztr <- std_ap(sc,Xtr); Zte <- std_ap(sc,Xte)
logit_model <- glm(y ~ ., data=data.frame(Ztr, y=as.integer(ytr=="mal")), family=binomial)
p_lr <- predict(logit_model, data.frame(Zte), type="response")
cat(sprintf("logistic regression: test AUC %.3f, Brier %.3f  |  black box: AUC %.3f, Brier %.3f\n",
    auc_of(yte,p_lr), mean((p_lr-(yte=="mal"))^2), auc_te, brier))
i <- which.min(abs(p_te-0.5)); z <- Zte[i,]; b <- coef(logit_model)
contrib <- sort(b[-1]*z)
cat(sprintf("patient %d: true class %s; black-box p = %.2f; logistic p = %.2f\n", i, as.character(yte[i]), p_te[i], p_lr[i]))
cat(sprintf("log-odds = intercept %.2f + sum of contributions %.2f = %.2f\n", b[1], sum(b[-1]*z), b[1]+sum(b[-1]*z)))
shown <- contrib                                                            # all nine features (only nine, so show every one)
par(mar=c(4,9,2,1)); barplot(shown, horiz=TRUE, las=1, col=ifelse(shown<0,BLUE,ORANGE), cex.names=0.7,
  xlab="contribution to the log-odds of malignancy", main=sprintf("Why the logistic model gives patient %d p = %.2f", i, p_lr[i]), cex.main=0.8)
abline(v=0)



<p class="caption"><b>Figure 12.</b> Exact local explanation of one test patient under the logistic regression: each bar is a coefficient times the patient's standardised value, its contribution to the log-odds of malignancy. The patient is the one the black box found most ambiguous (p near 0.5): a high clump thickness pushes strongly towards malignant, low bare-nuclei and nucleoli scores towards benign; the sum of intercept and contributions gives the logit, and the logistic probability printed above.</p>

The logistic regression's test AUC is within a thousandth of the black box's — Rudin's point in miniature: on nine well-designed tabular features the interpretable model loses nothing and answers "why?" exactly. The case for a black box must be a measurable gain on the <em>deployment</em> population. Where the gain exists — images, signals, text, where features are learned (Unit 8) — post-hoc explanations such as saliency maps are the only option and are hypotheses about the model, not reasons.

### 5.4 Privacy revisited: de-identification, synthetic data, federated learning, differential privacy

Unit 1 showed that removing identifiers does not anonymise. Three further techniques answer different questions. **Synthetic data**: a generative model trained on the real records samples new ones with the same joint distribution; a generator fitted to few records can memorise them, and without a formal guarantee its privacy is an empirical claim to be tested by membership-inference attacks. **Federated learning** lets several hospitals train one model without sharing records: in McMahan and colleagues' <em>federated averaging</em> each site trains locally and sends only parameters to a server, which averages them weighted by sample size and sends the average back, round after round.<sup class="c"><a href="#r23">[23]</a></sup> Rieke and colleagues review its use in digital health and its open problems: sites with different distributions, unequal data quality, and parameters that can themselves leak information about training records — federated learning is a governance tool, not a privacy guarantee.<sup class="c"><a href="#r24">[24]</a></sup> The chunk simulates one round on three "hospitals" carved from the training set.



In [ ]:
# Federated averaging, one round: three hospitals fit a logistic regression locally; only coefficients and summary statistics travel.
set.seed(0); site <- sample(0:2, length(ytr), replace=TRUE, prob=c(0.5,0.3,0.2))
ns <- sapply(0:2, function(s) sum(site==s))
means <- t(sapply(0:2, function(s) colMeans(Xtr[site==s,]))); vars <- t(sapply(0:2, function(s) apply(Xtr[site==s,],2,var)))
mu_g <- colSums(ns*means)/sum(ns)                                            # global scaler from site summaries only
sig_g <- sqrt(colSums(ns*(vars + (means - matrix(mu_g,3,ncol(Xtr),byrow=TRUE))^2))/sum(ns))
Zg <- function(Mx) scale(Mx, mu_g, sig_g)
local <- lapply(0:2, function(s) glm(y~., data=data.frame(Zg(Xtr[site==s,]), y=as.integer(ytr[site==s]=="mal")), family=binomial))
coef_fed <- as.numeric(sapply(local, coef) %*% ns / sum(ns))                 # server averages coefficients weighted by n
Zte_g <- cbind(1, Zg(Xte)); p_fed <- plogis(Zte_g %*% coef_fed)
central <- glm(y~., data=data.frame(Zg(Xtr), y=as.integer(ytr=="mal")), family=binomial)
rows <- data.frame(model=c(sprintf("hospital %d alone (n = %d)", 1:3, ns), "federated average (1 round)", "centralised (all pooled)"),
  test_AUC=c(sapply(0:2, function(s) auc_of(yte, plogis(Zte_g %*% coef(local[[s+1]])))),
             auc_of(yte, as.numeric(p_fed)), auc_of(yte, predict(central, data.frame(Zg(Xte)), type="response"))))
rows$test_AUC <- round(rows$test_AUC,4); print(rows, row.names=FALSE)



The federated model, built without any record leaving its hospital, matches the centralised one to within about a thousandth of AUC and beats the smallest hospital's own model. One round sufficed because the sites share a distribution and the model is convex; real federations need many rounds. What did travel — nine means, nine variances, ten coefficients per site — raises the question differential privacy answers.

**Differential privacy** is the only one of these techniques with a mathematical guarantee: a randomised algorithm \(\mathcal{M}\) is \(\varepsilon\)-differentially private if for every pair of datasets \(D, D'\) differing in one record and every output set \(S\), \(P(\mathcal{M}(D) \in S) \le e^{\varepsilon}\, P(\mathcal{M}(D') \in S)\) — the output barely changes whether or not any one person is in the data.<sup class="c"><a href="#r25">[25]</a></sup> The basic mechanism adds noise calibrated to the <em>sensitivity</em> of the query, the most one record can change it: for a count, Laplace noise of scale \(1/\varepsilon\). The budget \(\varepsilon\) is spent with every query and the guarantee composes. The chunk releases one count under three budgets.



In [ ]:
# The Laplace mechanism: releasing 'how many malignant cases are in the test set?' under three privacy budgets.
rlaplace <- function(n, b){ u <- runif(n)-0.5; -b*sign(u)*log(1-2*abs(u)) }
true_count <- sum(yte=="mal"); set.seed(0)
plot.new(); plot.window(xlim=c(true_count-40, true_count+40), ylim=c(0,0.9)); axis(1); axis(2)
cols <- c(GREY, ORANGE, BLUE); labs <- character()
for (j in seq_along(c(0.1,1,10))) { eps <- c(0.1,1,10)[j]
  rel <- true_count + rlaplace(5000, 1/eps); h <- hist(rel, breaks=seq(true_count-200,true_count+200,1), plot=FALSE)
  keep <- h$mids >= true_count-40 & h$mids <= true_count+40
  lines(h$mids[keep], h$density[keep], col=cols[j], lwd=2)
  labs[j] <- sprintf("epsilon = %g: within +/-%.0f (95%%)", eps, quantile(abs(rel-true_count),0.95)) }
abline(v=true_count, lty=2); legend("topleft", labs, col=cols, lwd=2, bty="n", cex=0.75)
title(sprintf("Differential privacy on a count (true = %d): smaller epsilon, stronger guarantee, noisier answer", true_count), cex.main=0.8)
mtext("released count", 1, line=2.2, cex=0.8)



<p class="caption"><b>Figure 13.</b> The count released by the Laplace mechanism at three budgets. At \(\varepsilon = 10\) the value is almost always within a unit or two of the truth and the guarantee is weak; at \(\varepsilon = 0.1\) the answer is useless for a dataset this size (off by tens) but no individual's presence can be inferred. The privacy–utility trade-off becomes one number that can be budgeted and audited.</p>

Differential privacy is applied to released statistics (the United States census since 2020), to model training (noise on gradient updates) and to synthetic-data generators. Its cost is accuracy, especially for small groups — where fairness audits need precision most; that tension is an open problem.

### 5.5 Governance and regulation

Four bodies of rules frame the work; Unit 1 introduced the first two.

**Data protection.** In Chile, Law 19.628 (1999) on the protection of private life; Law 20.584 (2012) on patients' rights, which makes the clinical record confidential; and Law 21.719, published in December 2024 and in force from 1 December 2026, which aligns the regime with the GDPR, creates the Agencia de Protección de Datos Personales with powers of inspection and sanction, and classifies health and genetic data as sensitive (see Online resources). Students graduating in 2027 will practise under the new law from their first day.

**Medical-device regulation.** Software that informs a clinical decision is a medical device: regulated by the FDA in the United States; in the European Union by the Medical Device Regulation (2017/745) and the <strong>Artificial Intelligence Act</strong> (Regulation 2024/1689), which classifies AI in medical devices as <em>high-risk</em> with obligations of risk management, data governance, documentation, transparency, human oversight, accuracy, robustness and cybersecurity, phased in 2025–2027 (see Online resources); in Chile by the Instituto de Salud Pública (ISP) under the Sanitary Code and Decree 825/1998, with a national AI policy and an AI bill following the European risk-based structure. A model deployed without sanitary registration is an unregistered medical device, whatever its AUC.

**Ethical guidance.** The WHO's 2021 guidance sets out six principles — autonomy; well-being, safety and the public interest; transparency, explainability and intelligibility; responsibility and accountability; inclusiveness and equity; responsive and sustainable AI (see Online resources). Each is instantiated here: model card and reporting standards (transparency), decision curve and cost-based threshold (safety), fairness audit (equity), differential privacy and consent (autonomy), the checklist (accountability).

**Research integrity and open science.** The <strong>FAIR principles</strong> ask that data and metadata be Findable, Accessible, Interoperable and Reusable — persistent identifiers, rich metadata, clearly conditioned access, standard vocabularies — under access controls when sensitive.<sup class="c"><a href="#r26">[26]</a></sup> Sandve and colleagues' ten rules for reproducible computational research — track how every result was produced, archive program versions, record seeds, keep raw data behind plots, give public access to scripts — are the design principles of this notebook: every number is reproduced by a chunk with a fixed seed, and the notebook is public.<sup class="c"><a href="#r27">[27]</a></sup> Authorship follows the same logic: whoever contributed substantially and can take responsibility is an author. A model trained by a student on a hospital's data and published without the hospital's knowledge violates the data-use agreement, the ethics approval and the norms of authorship at once.

### 5.6 A professional checklist for a bioengineer deploying a learning system

The course closes with sixteen questions to answer before, during and after deploying any system that learns from data about people.

<div class="box key"><h4>Before building</h4>
<ol>
<li><strong>Purpose.</strong> Which decision will the system inform, who makes it, and what happens to a patient when it is wrong in each direction?</li>
<li><strong>Label.</strong> Does the label measure what we intend, or a proxy that encodes an inequity? Who assigned it, with what error rate?</li>
<li><strong>Lawful basis and consent.</strong> Under which legal basis and ethics approval are the data used, and is this project within that purpose?</li>
<li><strong>Representativeness.</strong> From which sites, scanners, years and populations do the data come; which groups are absent?</li>
</ol></div>

<div class="box key"><h4>While building</h4>
<ol start="5">
<li><strong>Leakage.</strong> Is the test set locked, is every pre-processing step inside the cross-validation, are hyper-parameters and family chosen in nested loops?</li>
<li><strong>Calibration and threshold.</strong> Are the outputs calibrated probabilities, at which prevalence, and was the threshold chosen from clinical costs with clinicians?</li>
<li><strong>Utility.</strong> Does a decision curve show net benefit over the default strategies across plausible thresholds?</li>
<li><strong>Fairness.</strong> Which criterion is appropriate and why; does the model satisfy it across the subgroups that matter, with confidence intervals?</li>
<li><strong>Explanation.</strong> Could an interpretable model do the job? If not, what explanations will users receive and how were they validated?</li>
<li><strong>Privacy.</strong> Pseudonymised data, logged access, model checked for memorisation; is federated or differentially private training warranted?</li>
</ol></div>

<div class="box key"><h4>Before and after deployment</h4>
<ol start="11">
<li><strong>External validation.</strong> Has the model been evaluated on data from the deployment site and period?</li>
<li><strong>Documentation.</strong> Model card, model-facts label, and a report following TRIPOD+AI, DECIDE-AI or CONSORT-AI?</li>
<li><strong>Regulation.</strong> Is it a medical device, registered with the ISP (or FDA / EU notified body), meeting the AI Act's high-risk obligations where applicable?</li>
<li><strong>Human oversight and the five rights.</strong> Who can override it, how is that recorded, and does it reach the right person in the right format at the right time?</li>
<li><strong>Monitoring and withdrawal.</strong> Who monitors performance, calibration and subgroup error rates; what triggers retraining or withdrawal; who is accountable?</li>
<li><strong>Reproducibility.</strong> Are code, seeds, environment and, where lawful, data archived so that every number can be regenerated?</li>
</ol></div>

A system for which these questions cannot be answered is not ready, whatever its AUC. The methods of Units 2–8 make the answers possible; this unit is about having to give them.

## 6. A panorama of current applications, for Workshop 6

For Workshop 6, fifteen systems or studies, each with one primary reference and its <em>level of evidence</em> (Section 3.3). Pick one row, read the paper with the checklist of Section 5.6, and report which of its questions the paper answers.

| # | Domain | System or study | What it does (method of this course) | Evidence level | Ref. |
|---|---|---|---|---|---|
| 1 | Imaging — ophthalmology | IDx-DR, autonomous diabetic-retinopathy screening | CNN classifier on fundus photographs (U8) | Prospective pivotal trial in primary care; FDA authorisation for autonomous use, 2018 | <sup class="c"><a href="#r28">[28]</a></sup> |
| 2 | Imaging — radiology, pathology, dermatology | Deep-learning classifiers compared with specialists | CNNs, transfer learning (U8) | Meta-analysis of 82 studies: equivalence on average; 25 externally validated; high risk of bias | <sup class="c"><a href="#r10">[10]</a></sup> |
| 3 | Imaging — dermatology | Dermatologist-level skin-lesion classification | CNN fine-tuned from ImageNet | Retrospective reader study against dermatologists on biopsy-proven lesions | <sup class="c"><a href="#r28">[28]</a></sup> |
| 4 | Imaging — all specialties | Design quality of AI-versus-clinician studies | — | Systematic review: 81 non-randomised studies, 2 trials; over-claiming abstracts | <sup class="c"><a href="#r11">[11]</a></sup> |
| 5 | Signals — cardiology, wearables | AI-ECG and smartwatch rhythm classification | CNNs on ECG and photoplethysmography (U6/U8) | Large retrospective validations; FDA-cleared wearables; prospective evaluation of smartwatch notifications | <sup class="c"><a href="#r28">[28]</a></sup> |
| 6 | EHR prediction | Proprietary sepsis model deployed in hundreds of US hospitals | Penalised logistic regression on record variables (U2) | External validation after deployment: sensitivity 33 %, PPV 12 % — far below development claims | <sup class="c"><a href="#r33">[33]</a></sup> |
| 7 | EHR prediction | Sepsis Watch (Duke University Hospital) with a model-facts label | Deep model on streaming record data with a one-page facts label | Deployed with prospective monitoring; the reference example of user-facing documentation | <sup class="c"><a href="#r31">[31]</a></sup> |
| 8 | EHR prediction | Deterioration, readmission and mortality prediction | Regression, tree ensembles, deep networks (U2, U4, U6) | Review: strong retrospective results; barriers are pipelines, workflow, outcome evaluation | <sup class="c"><a href="#r30">[30]</a></sup> |
| 9 | EHR prediction | COVID-19 diagnostic and prognostic models | Mostly logistic regression and tree ensembles (U2, U4) | Living review: hundreds of models, almost all at high risk of bias, none recommended | <sup class="c"><a href="#r19">[19]</a></sup> |
| 10 | Population health | Commercial algorithm allocating care-management resources | Regression predicting cost as a proxy for need (U2) | Deployed at scale; audit found large racial bias through the label | <sup class="c"><a href="#r29">[29]</a></sup> |
| 11 | Clinical text | Information extraction from clinical notes | Rule-based and bag-of-words pipelines (Section 2.8) | Review of 263 studies; mostly single-institution, rarely externally validated | <sup class="c"><a href="#r7">[7]</a></sup> |
| 12 | Multi-site imaging | Federated learning consortia for medical imaging | Federated averaging across hospitals (Section 5.4) | Review of early deployments; open problems of heterogeneity and privacy | <sup class="c"><a href="#r24">[24]</a></sup> |
| 13 | Structural biology | AlphaFold2 protein-structure prediction | Attention network on sequences and alignments (BGM324) | Blind assessment (CASP14); near-experimental accuracy; public structure database | <sup class="c"><a href="#r34">[34]</a></sup> |
| 14 | Drug discovery | QSAR classification of urease and TRPV1 inhibitors | Tree ensembles on descriptors; repurposing; docking and simulation (U4; BGM324) | Cross-validated computational studies; candidates prioritised for testing | <sup class="c"><a href="#r8">[8]</a>,<a href="#r9">[9]</a></sup> |
| 15 | Language models | Med-PaLM: large language model on medical question answering | Transformer language model (U1) | Benchmark evaluation with physician rating; no clinical evaluation | <sup class="c"><a href="#r35">[35]</a></sup> |

Three patterns: evidence level is independent of method sophistication — the simplest model (row 6) is the most widely deployed and the one shown to fail; the systems with prospective evidence (rows 1, 5, 7) are narrow and were evaluated at the level of the decision; and the audits and reviews (rows 4, 6, 9, 10) are the most instructive, because they show which checklist questions were not asked.

## 7. Summary of the course

You should now be able to explain, without notes:

- KDD and CRISP-DM; the four task families; why most work is before and after the algorithm.
- Support, confidence, lift; Apriori's join and prune; why many rules produce spurious ones and how a permutation null and Bonferroni expose them.
- Isolation forests ("different" is not "diseased"); the bag of words and what it loses.
- The five imaging tasks; what makes medical images different; Dice and IoU; the evidence base; how a change of scanner breaks an honestly validated model.
- MYCIN and INTERNIST-1 to decision support; the five rights; likelihood ratios and Fagan's nomogram; why a classifier's probability is tied to its training prevalence.
- The honest diagnostic model: nested cross-validation, calibration, decision curve, cost-based threshold, subgroup check, model card; the three reporting standards; why most models never reach practice.
- The data life cycle; the three fairness criteria and the impossibility result; explanation and its limits; synthetic data, federated averaging, differential privacy; the legal frame, WHO principles, FAIR, reproducibility; the sixteen-question checklist.

## 8. Self-assessment

<details><summary>1. Which part of the course serves each KDD adjective — valid, novel, useful, understandable — and which does a black box with AUC 0.99 fail by default?</summary>Valid: Supplement S1. Novel: domain knowledge (Section 2.1). Useful: decision curves and the five rights (Sections 4.2, 4.4). Understandable: Section 5.3 — which the black box fails by default.</details>

<details><summary>2. Of 200 patients, 80 have hypertension, 60 take a statin, 36 both. Support, confidence and lift of "hypertension ⇒ statin"?</summary>Support 36/200 = 0.18; confidence 36/80 = 0.45; lift 0.45/0.30 = 1.5: a positive association, whose significance needs a test and whose causality cannot be known from these data.</details>

<details><summary>3. Why was {HTN, DM, ACEi} pruned without being counted, and what changes without pruning?</summary>Its subset {DM, ACEi} was infrequent, so by the Apriori property it cannot be frequent. Without pruning the result is unchanged but many more candidates are counted per level — and on a real database the number of candidates is what makes the problem intractable.</details>

<details><summary>4. A colleague mines 10,000 drug–event pairs and reports every pair with lift above 2. What is wrong?</summary>With 10,000 tests many pairs exceed any fixed lift by chance, especially rare ones (Figure 3). Test each pair, correct for the number of tests (Bonferroni or false-discovery rate), treat rare pairs with caution, and confirm survivors in independent data and by clinical review.</details>

<details><summary>5. |S| = 500 pixels, |G| = 400, overlap 300. Compute Dice and IoU and check the identity.</summary>Dice = 600/900 = 0.667; IoU = 300/600 = 0.500; 2 × 0.5/1.5 = 0.667. ✓</details>

<details><summary>6. In Figure 6 accuracy collapsed on a "different scanner" although cross-validated and test accuracies agreed. Why did neither warn?</summary>Both were computed on the training distribution and say nothing about another. Only evaluation on data from the other scanner, or training with augmentation that includes the variation, addresses the shift.</details>

<details><summary>7. Sensitivity 0.95, specificity 0.80: LR⁺ and post-test probability after a positive result at pre-test 0.10 and 0.50?</summary>LR⁺ = 0.95/0.20 = 4.75. Odds 0.111 → 0.528 → p = 0.35. Odds 1 → 4.75 → p = 0.83. The same result yields 35 % or 83 % depending on who was tested.</details>

<details><summary>8. The code picked the family with the highest outer-fold AUC. Is that AUC unbiased? Is the test-set AUC?</summary>The selected outer AUC is slightly optimistic — the maximum of three noisy estimates. The test-set AUC is unbiased for the whole procedure, including the selection, because the test set played no role in any choice.</details>

<details><summary>9. In Figure 9, what does a net benefit above "biopsy everyone" at p_t = 0.20 mean, and why do all curves meet near p_t = 0?</summary>At p_t = 0.20 a clinician accepts four unnecessary biopsies per cancer; a higher curve means acting on the model finds more cancers net of harm at that exchange rate. Near p_t = 0 a false positive carries no harm, so biopsying everyone is optimal.</details>

<details><summary>10. The older group was flagged much more often. Unfair? Which criterion fails, which hold, and why not all three?</summary>Demographic parity fails, appropriately, because malignancy is more common in the older group; equalised odds and calibration within groups hold approximately. With different base rates and an imperfect classifier, calibration and equal error rates cannot both hold exactly, so a justified choice is unavoidable.</details>

<details><summary>11. In the federated chunk, what left each hospital, and why does differential privacy remain relevant?</summary>Sample size, nine means, nine variances and ten coefficients — functions of the patient data that can leak information about individuals (a small site's mean shifts with one extreme patient; coefficients can memorise rare records). A formal guarantee on the transmitted quantities is needed for federation to be privacy-preserving rather than merely data-local.</details>

<details><summary>12. A hospital wants the capstone model deployed next month. Which four checklist questions does this notebook answer inadequately?</summary>External validation (Q11): internal validation on single-site data only. Fairness (Q8): no demographic variables. Regulation (Q13): a medical device without ISP registration. Oversight and monitoring (Q14–15): no plan for overrides, monitoring or withdrawal. Lawful basis for the local data (Q3) is a fifth.</details>

## Glossary

<dl class="gloss">
<dt>Apriori property</dt><dd>Every subset of a frequent itemset is frequent; the basis of the prune step.</dd>
<dt>Association rule</dt><dd>A ⇒ B between disjoint itemsets; co-occurrence, not causation.</dd>
<dt>Bonferroni correction</dt><dd>Dividing the significance level by the number of tests performed.</dd>
<dt>Brier score</dt><dd>Mean squared difference between predicted probability and the 0/1 outcome; lower is better.</dd>
<dt>Calibration</dt><dd>Agreement between predicted probabilities and observed frequencies.</dd>
<dt>CDSS</dt><dd>Clinical decision support system.</dd>
<dt>Confidence / support / lift</dt><dd>P(B | A) / P(A ∪ B) / P(A, B) divided by P(A)P(B), estimated from transaction frequencies; lift is 1 under independence.</dd>
<dt>CRISP-DM</dt><dd>Business understanding, data understanding, data preparation, modelling, evaluation, deployment.</dd>
<dt>Decision-curve analysis</dt><dd>Net benefit of a model across threshold probabilities against treating everyone and treating no one.</dd>
<dt>Demographic parity / equalised odds / calibration within groups</dt><dd>Equal positive-decision rate / equal sensitivity and false-positive rate / equal meaning of a predicted probability across groups; incompatible in general when base rates differ.</dd>
<dt>Dice / IoU</dt><dd>2|S ∩ G|/(|S| + |G|) and |S ∩ G|/|S ∪ G|; Dice = 2 IoU/(1 + IoU).</dd>
<dt>Differential privacy</dt><dd>The output distribution changes by at most a factor e^ε when one record is added or removed.</dd>
<dt>Domain shift</dt><dd>A change in input distribution between training and deployment.</dd>
<dt>Federated learning</dt><dd>Training across data holders by exchanging parameters rather than records.</dd>
<dt>Isolation forest</dt><dd>Anomaly detector: points isolated by few random splits are anomalous. Needs no labels.</dd>
<dt>KDD</dt><dd>Knowledge discovery in databases; data mining is one of its steps.</dd>
<dt>Likelihood ratio</dt><dd>LR⁺ = sensitivity/(1 − specificity); LR⁻ = (1 − sensitivity)/specificity; multiplies pre-test odds.</dd>
<dt>Model card</dt><dd>Structured statement of a model's details, intended use, data, metrics by subgroup and caveats.</dd>
<dt>Nested cross-validation</dt><dd>Inner loop for hyper-parameters inside each training fold of an outer loop for performance.</dd>
<dt>Net benefit</dt><dd>TP/n − FP/n × p_t/(1 − p_t).</dd>
<dt>Partial dependence / permutation importance</dt><dd>Average prediction as one feature varies / performance loss when one feature is shuffled.</dd>
<dt>TRIPOD+AI, DECIDE-AI, CONSORT-AI</dt><dd>Reporting standards for model development and validation, early clinical evaluation, and randomised trials with an AI component.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Fayyad U, Piatetsky-Shapiro G, Smyth P. From data mining to knowledge discovery in databases. <em>AI Magazine</em>. 1996;17(3):37–54. <a href="https://ojs.aaai.org/aimagazine/index.php/aimagazine/article/view/1230">ojs.aaai.org/aimagazine/…/view/1230</a> (publisher landing page; the DOI 10.1609/aimag.v17i3.1230 given by the publisher is not indexed in Crossref and could not be verified at the time of writing)</li>
<li id="r2">Schröer C, Kruse F, Gómez JM. A systematic literature review on applying CRISP-DM process model. <em>Procedia Comput Sci</em>. 2021;181:526–534. <a href="https://doi.org/10.1016/j.procs.2021.01.199">doi:10.1016/j.procs.2021.01.199</a></li>
<li id="r3">Agrawal R, Imieliński T, Swami A. Mining association rules between sets of items in large databases. <em>Proc 1993 ACM SIGMOD Int Conf Management of Data</em>. 1993:207–216. <a href="https://doi.org/10.1145/170035.170072">doi:10.1145/170035.170072</a></li>
<li id="r4">Agrawal R, Srikant R. Fast algorithms for mining association rules in large databases. <em>Proc 20th Int Conf Very Large Data Bases (VLDB 1994)</em>. 1994:487–499. <a href="https://www.vldb.org/conf/1994/P487.PDF">vldb.org/conf/1994/P487.PDF</a> (conference paper without a DOI)</li>
<li id="r5">Brin S, Motwani R, Silverstein C. Beyond market baskets: generalizing association rules to correlations. <em>Proc 1997 ACM SIGMOD Int Conf Management of Data</em>. 1997:265–276. <a href="https://doi.org/10.1145/253260.253327">doi:10.1145/253260.253327</a></li>
<li id="r6">Liu FT, Ting KM, Zhou Z-H. Isolation forest. <em>Proc 8th IEEE Int Conf Data Mining (ICDM 2008)</em>. 2008:413–422. <a href="https://doi.org/10.1109/ICDM.2008.17">doi:10.1109/ICDM.2008.17</a></li>
<li id="r7">Wang Y, Wang L, Rastegar-Mojarad M, et al. Clinical information extraction applications: a literature review. <em>J Biomed Inform</em>. 2018;77:34–49. <a href="https://doi.org/10.1016/j.jbi.2017.11.011">doi:10.1016/j.jbi.2017.11.011</a></li>
<li id="r8">Morales N, Valdés-Muñoz E, González J, et al. Machine learning-driven classification of urease inhibitors leveraging physicochemical properties as effective filter criteria. <em>Int J Mol Sci</em>. 2024;25(8):4303. <a href="https://doi.org/10.3390/ijms25084303">doi:10.3390/ijms25084303</a></li>
<li id="r9">González J, Palma JM, Benso B, et al. Integrative computational approaches for TRPV1 ion channel inhibitor discovery: an integrated machine learning, drug repurposing and molecular simulation approach. <em>J Chem Inf Model</em>. 2025;65(17):8957–8968. <a href="https://doi.org/10.1021/acs.jcim.5c00991">doi:10.1021/acs.jcim.5c00991</a></li>
<li id="r10">Liu X, Faes L, Kale AU, et al. A comparison of deep learning performance against health-care professionals in detecting diseases from medical imaging: a systematic review and meta-analysis. <em>Lancet Digit Health</em>. 2019;1(6):e271–e297. <a href="https://doi.org/10.1016/S2589-7500(19)30123-2">doi:10.1016/S2589-7500(19)30123-2</a></li>
<li id="r11">Nagendran M, Chen Y, Lovejoy CA, et al. Artificial intelligence versus clinicians: systematic review of design, reporting standards, and claims of deep learning studies. <em>BMJ</em>. 2020;368:m689. <a href="https://doi.org/10.1136/bmj.m689">doi:10.1136/bmj.m689</a></li>
<li id="r12">Miller RA, Pople HE, Myers JD. INTERNIST-1, an experimental computer-based diagnostic consultant for general internal medicine. <em>N Engl J Med</em>. 1982;307(8):468–476. <a href="https://doi.org/10.1056/NEJM198208193070803">doi:10.1056/NEJM198208193070803</a></li>
<li id="r13">Fagan TJ. Nomogram for Bayes's theorem. <em>N Engl J Med</em>. 1975;293(5):257. <a href="https://doi.org/10.1056/NEJM197507312930513">doi:10.1056/NEJM197507312930513</a></li>
<li id="r14">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r15">Mitchell M, Wu S, Zaldivar A, et al. Model cards for model reporting. <em>Proc Conf Fairness, Accountability, and Transparency (FAT* 2019)</em>. 2019:220–229. <a href="https://doi.org/10.1145/3287560.3287596">doi:10.1145/3287560.3287596</a></li>
<li id="r16">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r17">Liu X, Cruz Rivera S, Moher D, Calvert MJ, Denniston AK; SPIRIT-AI and CONSORT-AI Working Group. Reporting guidelines for clinical trial reports for interventions involving artificial intelligence: the CONSORT-AI extension. <em>Nat Med</em>. 2020;26(9):1364–1374. <a href="https://doi.org/10.1038/s41591-020-1034-x">doi:10.1038/s41591-020-1034-x</a></li>
<li id="r18">Vasey B, Nagendran M, Campbell B, et al. Reporting guideline for the early-stage clinical evaluation of decision support systems driven by artificial intelligence: DECIDE-AI. <em>Nat Med</em>. 2022;28(5):924–933. <a href="https://doi.org/10.1038/s41591-022-01772-9">doi:10.1038/s41591-022-01772-9</a></li>
<li id="r19">Wynants L, Van Calster B, Collins GS, et al. Prediction models for diagnosis and prognosis of covid-19: systematic review and critical appraisal. <em>BMJ</em>. 2020;369:m1328. <a href="https://doi.org/10.1136/bmj.m1328">doi:10.1136/bmj.m1328</a></li>
<li id="r20">Chouldechova A. Fair prediction with disparate impact: a study of bias in recidivism prediction instruments. <em>Big Data</em>. 2017;5(2):153–163. <a href="https://doi.org/10.1089/big.2016.0047">doi:10.1089/big.2016.0047</a></li>
<li id="r21">Kleinberg J, Mullainathan S, Raghavan M. Inherent trade-offs in the fair determination of risk scores. <em>Proc 8th Innovations in Theoretical Computer Science Conference (ITCS 2017)</em>, LIPIcs 67:43:1–43:23. arXiv preprint. <a href="https://doi.org/10.48550/arXiv.1609.05807">doi:10.48550/arXiv.1609.05807</a></li>
<li id="r22">Rudin C. Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. <em>Nat Mach Intell</em>. 2019;1(5):206–215. <a href="https://doi.org/10.1038/s42256-019-0048-x">doi:10.1038/s42256-019-0048-x</a></li>
<li id="r23">McMahan HB, Moore E, Ramage D, Hampson S, Agüera y Arcas B. Communication-efficient learning of deep networks from decentralized data. <em>Proc 20th Int Conf Artificial Intelligence and Statistics (AISTATS 2017), PMLR</em>. 2017;54:1273–1282. <a href="https://proceedings.mlr.press/v54/mcmahan17a.html">proceedings.mlr.press/v54/mcmahan17a.html</a> (PMLR proceedings have no DOI)</li>
<li id="r24">Rieke N, Hancox J, Li W, et al. The future of digital health with federated learning. <em>npj Digit Med</em>. 2020;3:119. <a href="https://doi.org/10.1038/s41746-020-00323-1">doi:10.1038/s41746-020-00323-1</a></li>
<li id="r25">Dwork C, Roth A. The algorithmic foundations of differential privacy. <em>Found Trends Theor Comput Sci</em>. 2014;9(3–4):211–407. <a href="https://doi.org/10.1561/0400000042">doi:10.1561/0400000042</a></li>
<li id="r26">Wilkinson MD, Dumontier M, Aalbersberg IJ, et al. The FAIR Guiding Principles for scientific data management and stewardship. <em>Sci Data</em>. 2016;3:160018. <a href="https://doi.org/10.1038/sdata.2016.18">doi:10.1038/sdata.2016.18</a></li>
<li id="r27">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r28">Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25(1):44–56. <a href="https://doi.org/10.1038/s41591-018-0300-7">doi:10.1038/s41591-018-0300-7</a></li>
<li id="r29">Obermeyer Z, Powers B, Vogeli C, Mullainathan S. Dissecting racial bias in an algorithm used to manage the health of populations. <em>Science</em>. 2019;366(6464):447–453. <a href="https://doi.org/10.1126/science.aax2342">doi:10.1126/science.aax2342</a></li>
<li id="r30">Rajkomar A, Dean J, Kohane I. Machine learning in medicine. <em>N Engl J Med</em>. 2019;380(14):1347–1358. <a href="https://doi.org/10.1056/NEJMra1814259">doi:10.1056/NEJMra1814259</a></li>
<li id="r31">Sendak MP, Gao M, Brajer N, Balu S. Presenting machine learning model information to clinical end users with model facts labels. <em>npj Digit Med</em>. 2020;3:41. <a href="https://doi.org/10.1038/s41746-020-0253-3">doi:10.1038/s41746-020-0253-3</a></li>
<li id="r32">Char DS, Shah NH, Magnus D. Implementing machine learning in health care — addressing ethical challenges. <em>N Engl J Med</em>. 2018;378(11):981–983. <a href="https://doi.org/10.1056/NEJMp1714229">doi:10.1056/NEJMp1714229</a></li>
<li id="r33">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065–1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r34">Jumper J, Evans R, Pritzel A, et al. Highly accurate protein structure prediction with AlphaFold. <em>Nature</em>. 2021;596(7873):583–589. <a href="https://doi.org/10.1038/s41586-021-03819-2">doi:10.1038/s41586-021-03819-2</a></li>
<li id="r35">Singhal K, Azizi S, Tu T, et al. Large language models encode clinical knowledge. <em>Nature</em>. 2023;620(7972):172–180. <a href="https://doi.org/10.1038/s41586-023-06291-2">doi:10.1038/s41586-023-06291-2</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package (ships the Wisconsin BreastCancer data used here). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group, TU Wien. R package (the SVM and naive-Bayes implementations). <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r39">Liaw A, Wiener M. Classification and regression by randomForest. <em>R News</em>. 2002;2(3):18–22. <a href="https://doi.org/10.32614/CRAN.package.randomForest">doi:10.32614/CRAN.package.randomForest</a></li>
<li id="r40">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r41">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002 (the reference for <code>nnet</code>, <code>lda</code> and the R modelling functions used throughout). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 14.2 for association rules and the Apriori algorithm; Chapter 7 for model assessment. Free PDF from the authors' site.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em> (ISLR). 2nd ed. Springer; 2021. — Chapter 5 for resampling and Chapter 4 for classification thresholds; the R companion to the whole course. Free PDF at statlearning.com.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Chapter 1.5 for decision theory (costs, rejection, the odds form of Bayes).</li>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. — Chapter 27 ("Philosophy, ethics and safety of AI") and Chapter 12 for probabilistic reasoning.</li>
<li>Osheroff JA, Teich JM, Levick D, et al. <em>Improving Outcomes with Clinical Decision Support: An Implementer's Guide</em>. 2nd ed. HIMSS; 2012. — The source of the "five rights" of CDSS.</li>
<li>Barocas S, Hardt M, Narayanan A. <em>Fairness and Machine Learning: Limitations and Opportunities</em>. MIT Press; 2023. — The reference text for Section 5.2. Free online at fairmlbook.org.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>CRAN task views "MachineLearning" and "Cluster" — <a href="https://cran.r-project.org/web/views/">cran.r-project.org/web/views</a>; the R counterparts of the scikit-learn user guide's "Probability calibration", "Inspection" and "Novelty and outlier detection" sections.</li>
<li>TRIPOD statement and TRIPOD+AI checklist — <a href="https://www.tripod-statement.org">tripod-statement.org</a>.</li>
<li>EQUATOR Network (reporting guidelines for health research, including CONSORT-AI, SPIRIT-AI, DECIDE-AI) — <a href="https://www.equator-network.org">equator-network.org</a>.</li>
<li>FDA, Artificial Intelligence-Enabled Medical Devices list — <a href="https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-enabled-medical-devices">fda.gov</a>.</li>
<li>Instituto de Salud Pública de Chile, Dispositivos Médicos — <a href="https://www.ispch.cl">ispch.cl</a>.</li>
<li>Biblioteca del Congreso Nacional de Chile, Ley Chile — <a href="https://www.bcn.cl/leychile">bcn.cl/leychile</a>; consolidated texts of Laws 19.628, 20.120, 20.584 and 21.719 (the last published 13 December 2024, in force 1 December 2026).</li>
<li>WHO guidance on ethics and governance of AI for health (2021) — <a href="https://www.who.int/publications/i/item/9789240029200">who.int/publications/i/item/9789240029200</a>.</li>
<li>Regulation (EU) 2024/1689 (Artificial Intelligence Act), Official Journal of the European Union — <a href="https://eur-lex.europa.eu/eli/reg/2024/1689/oj">eur-lex.europa.eu/eli/reg/2024/1689/oj</a>.</li>
</ul>
